# Retinal DR Locality Kill Test v0.1

## Purpose

Determine whether the eye-local DR severity signal detected by the frozen
ResNet-50 representation depends on spatially local pathological structure,
global photometric appearance, retinal anatomy, or residual acquisition cues.

The frozen baseline, endpoint definition, training patients, validation
patients and primary paired-difference evaluation protocol are not modified
in this notebook.

In [2]:
#@title 00. Load frozen baseline artifacts

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability"
)

BASELINE_ROOT = (
    PROJECT_ROOT
    / "06_Data_Records"
    / "Retinal_DR"
    / "Frozen_Representation_Baseline_v0.1"
)

KILL_TEST_ROOT = (
    PROJECT_ROOT
    / "06_Data_Records"
    / "Retinal_DR"
    / "Locality_Kill_Test_v0.1"
)

KILL_TEST_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

IMAGE_INDEX_PATH = (
    BASELINE_ROOT
    / "ResNet50_ImageNet1K_V2_Embedding_Index.csv"
)

EYE_INDEX_PATH = (
    BASELINE_ROOT
    / "ResNet50_ImageNet1K_V2_EyeMean_Index_and_Predictions.csv"
)

BASELINE_DECISION_PATH = (
    BASELINE_ROOT
    / "Frozen_Representation_Baseline_v0.1_Decision.json"
)

BASELINE_EVIDENCE_PATH = (
    BASELINE_ROOT
    / "Frozen_Representation_Baseline_v0.1_Evidence_Table.csv"
)

BASELINE_PAIRED_PREDICTIONS_PATH = (
    BASELINE_ROOT
    / "Validation_Ordinal_PairedDifference_Predictions.csv"
)

required_paths = [
    IMAGE_INDEX_PATH,
    EYE_INDEX_PATH,
    BASELINE_DECISION_PATH,
    BASELINE_EVIDENCE_PATH,
    BASELINE_PAIRED_PREDICTIONS_PATH,
]

missing_paths = [
    path
    for path in required_paths
    if not path.is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Missing frozen baseline artifacts:\n"
        + "\n".join(map(str, missing_paths))
    )

image_meta = pd.read_csv(
    IMAGE_INDEX_PATH
)

eye_meta_frozen = pd.read_csv(
    EYE_INDEX_PATH
)

baseline_evidence = pd.read_csv(
    BASELINE_EVIDENCE_PATH
)

baseline_pair_predictions = pd.read_csv(
    BASELINE_PAIRED_PREDICTIONS_PATH
)

with open(
    BASELINE_DECISION_PATH,
    "r",
    encoding="utf-8",
) as file:
    baseline_decision = json.load(file)

print("================ FROZEN BASELINE IMPORT ================")

print("Baseline decision:")
print(baseline_decision["decision"])

print("\nImage-level rows:")
print(len(image_meta))

print("\nEye-level rows:")
print(len(eye_meta_frozen))

print("\nValidation unequal-grade pairs:")
print(len(baseline_pair_predictions))

print("\nPrimary baseline locality result:")
print(
    baseline_decision[
        "primary_locality_result"
    ]
)

print("\nKill-test output root:")
print(KILL_TEST_ROOT)

assert (
    baseline_decision["decision"]
    == "PASS_BASELINE_ADVANCE_TO_LOCALITY_KILL_TEST"
)

assert len(image_meta) == 1600
assert len(eye_meta_frozen) == 800
assert len(baseline_pair_predictions) == 45

print("\nFrozen baseline loaded successfully.")

Mounted at /content/drive
================ FROZEN BASELINE IMPORT ================
Baseline decision:
PASS_BASELINE_ADVANCE_TO_LOCALITY_KILL_TEST

Image-level rows:
1600

Eye-level rows:
800

Validation unequal-grade pairs:
45

Primary baseline locality result:
{'correct_pairs': 31, 'total_pairs': 45, 'accuracy': 0.6888888888888889, 'exact_two_sided_95ci': [0.5335089700878829, 0.818341196066763], 'balanced_accuracy': 0.688259109311741, 'directional_auc': 0.7530364372469636, 'exact_one_sided_binomial_p': 0.008047180015637421}

Kill-test output root:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Locality_Kill_Test_v0.1

Frozen baseline loaded successfully.


In [2]:
#@title 01. Freeze kill-test protocol and reload frozen encoder

from pathlib import Path
from PIL import Image

import hashlib
import json
import platform

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision

from torchvision import transforms
from torchvision.models import (
    resnet50,
    ResNet50_Weights,
)
from torchvision.transforms import functional as TF
from torchvision.transforms.functional import InterpolationMode


RANDOM_SEED = 20260719


# ------------------------------------------------------------
# 1. Runtime
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("Device:", device)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0),
    )

assert device.type == "cuda", (
    "The kill test should be run with a GPU runtime."
)


# ------------------------------------------------------------
# 2. Reload the exact frozen encoder
# ------------------------------------------------------------

weights = ResNet50_Weights.IMAGENET1K_V2

official_preprocess = weights.transforms()

encoder = resnet50(
    weights=weights
)

embedding_dim = encoder.fc.in_features
encoder.fc = nn.Identity()

encoder.eval()

for parameter in encoder.parameters():
    parameter.requires_grad = False

encoder = encoder.to(device)

trainable_parameters = sum(
    parameter.numel()
    for parameter in encoder.parameters()
    if parameter.requires_grad
)

print("\nEncoder: ResNet-50 / ImageNet-1K V2")
print("Embedding dimension:", embedding_dim)
print("Trainable parameters:", trainable_parameters)

assert embedding_dim == 2048
assert trainable_parameters == 0


# ------------------------------------------------------------
# 3. Load the frozen clean image-level embedding cache
# ------------------------------------------------------------

BASELINE_IMAGE_EMBEDDING_PATH = (
    BASELINE_ROOT
    / "ResNet50_ImageNet1K_V2_Embeddings_float32.npy"
)

assert BASELINE_IMAGE_EMBEDDING_PATH.is_file(), (
    BASELINE_IMAGE_EMBEDDING_PATH
)

clean_image_embeddings = np.load(
    BASELINE_IMAGE_EMBEDDING_PATH
)

assert clean_image_embeddings.shape == (
    1600,
    embedding_dim,
)

assert clean_image_embeddings.dtype == np.float32
assert np.isfinite(clean_image_embeddings).all()

print(
    "\nClean cached embeddings:",
    clean_image_embeddings.shape,
)


# ------------------------------------------------------------
# 4. Freeze the perturbation protocol before evaluation
# ------------------------------------------------------------

kill_test_protocol = pd.DataFrame(
    [
        {
            "condition": "clean_cached",
            "status": "frozen baseline",
            "operation": (
                "No perturbation; reuse the sealed baseline "
                "image embeddings."
            ),
            "primarily_destroys": "Nothing",
            "primarily_preserves": (
                "All information surviving the original "
                "ResNet-50 preprocessing"
            ),
            "primary_question": (
                "Reference performance for all paired comparisons"
            ),
        },
        {
            "condition": "grayscale",
            "status": "pre-registered",
            "operation": (
                "Convert the 224x224 crop to grayscale and "
                "replicate it across three channels."
            ),
            "primarily_destroys": (
                "Chromatic information and colour differences"
            ),
            "primarily_preserves": (
                "Luminance, spatial anatomy and lesion morphology"
            ),
            "primary_question": (
                "Does eye-local severity direction require colour?"
            ),
        },
        {
            "condition": "strong_blur",
            "status": "pre-registered",
            "operation": (
                "Gaussian blur on the 224x224 crop; "
                "kernel=31, sigma=10."
            ),
            "primarily_destroys": (
                "Small lesions, sharp vascular edges and "
                "fine texture"
            ),
            "primarily_preserves": (
                "Global colour, illumination and coarse anatomy"
            ),
            "primary_question": (
                "Does the locality result depend on fine "
                "pathological structure?"
            ),
        },
        {
            "condition": "patch_shuffle_4x4",
            "status": "pre-registered",
            "operation": (
                "Shuffle sixteen 56x56 patches using one fixed "
                "patient-specific permutation."
            ),
            "primarily_destroys": (
                "Global retinal topology and spatial arrangement"
            ),
            "primarily_preserves": (
                "Local texture, local colour and patch statistics"
            ),
            "primary_question": (
                "Does the locality result require global "
                "anatomical organisation?"
            ),
        },
    ]
)

PROTOCOL_PATH = (
    KILL_TEST_ROOT
    / "Locality_Kill_Test_v0.1_Protocol.csv"
)

kill_test_protocol.to_csv(
    PROTOCOL_PATH,
    index=False,
)

display(kill_test_protocol)

print("\nProtocol saved:")
print(PROTOCOL_PATH)


# ------------------------------------------------------------
# 5. Build a preprocessing pipeline equivalent to the
#    official ImageNet-1K V2 preprocessing
# ------------------------------------------------------------

resize_transform = transforms.Resize(
    232,
    interpolation=InterpolationMode.BILINEAR,
    antialias=True,
)

crop_transform = transforms.CenterCrop(
    224
)

to_tensor_transform = transforms.ToTensor()

normalise_transform = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225],
)


def prepare_clean_crop(image):
    image = resize_transform(image)
    image = crop_transform(image)
    return image


def clean_tensor_from_pil(image):
    image = prepare_clean_crop(image)
    tensor = to_tensor_transform(image)
    tensor = normalise_transform(tensor)
    return tensor


# ------------------------------------------------------------
# 6. Verify that the custom clean path matches the official
#    weights preprocessing before defining perturbations
# ------------------------------------------------------------

comparison_rows = image_meta.iloc[:8]

maximum_preprocess_difference = 0.0

for row in comparison_rows.itertuples(index=False):
    image_path = Path(row.image_path_disk)

    with Image.open(image_path) as image:
        image = image.convert("RGB")

        official_tensor = official_preprocess(
            image
        )

        custom_tensor = clean_tensor_from_pil(
            image
        )

    difference = torch.max(
        torch.abs(
            official_tensor
            - custom_tensor
        )
    ).item()

    maximum_preprocess_difference = max(
        maximum_preprocess_difference,
        difference,
    )

print(
    "\nMaximum official-versus-custom "
    "preprocessing difference:",
    maximum_preprocess_difference,
)

assert maximum_preprocess_difference < 1e-5, (
    "Custom preprocessing does not reproduce the "
    "official ResNet-50 V2 preprocessing."
)

print("\nKill-test protocol and encoder setup passed.")

PyTorch: 2.11.0+cu128
TorchVision: 0.26.0+cu128
Device: cuda
GPU: Tesla T4
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 157MB/s]



Encoder: ResNet-50 / ImageNet-1K V2
Embedding dimension: 2048
Trainable parameters: 0

Clean cached embeddings: (1600, 2048)


,condition,status,operation,primarily_destroys,primarily_preserves,primary_question
0,clean_cached,frozen baseline,No perturbation; reuse the sealed baseline ima...,Nothing,All information surviving the original ResNet-...,Reference performance for all paired comparisons
1,grayscale,pre-registered,Convert the 224x224 crop to grayscale and repl...,Chromatic information and colour differences,"Luminance, spatial anatomy and lesion morphology",Does eye-local severity direction require colour?
2,strong_blur,pre-registered,"Gaussian blur on the 224x224 crop; kernel=31, ...","Small lesions, sharp vascular edges and fine t...","Global colour, illumination and coarse anatomy",Does the locality result depend on fine pathol...
3,patch_shuffle_4x4,pre-registered,Shuffle sixteen 56x56 patches using one fixed ...,Global retinal topology and spatial arrangement,"Local texture, local colour and patch statistics",Does the locality result require global anatom...



Protocol saved:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Locality_Kill_Test_v0.1/Locality_Kill_Test_v0.1_Protocol.csv

Maximum official-versus-custom preprocessing difference: 0.0

Kill-test protocol and encoder setup passed.


In [3]:
#@title 02. Extract and cache pre-registered perturbation embeddings

from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

import numpy as np
import pandas as pd
import torch


# ------------------------------------------------------------
# 1. Stable patient-specific seed
# ------------------------------------------------------------

def stable_patient_seed(
    patient_id,
    base_seed=RANDOM_SEED,
):
    text = (
        f"{base_seed}|{str(patient_id)}"
    ).encode("utf-8")

    digest = hashlib.sha256(
        text
    ).digest()

    return int.from_bytes(
        digest[:8],
        byteorder="little",
        signed=False,
    ) % (2**31 - 1)


# ------------------------------------------------------------
# 2. Deterministic 4x4 patch shuffle
# ------------------------------------------------------------

def shuffle_tensor_patches(
    image_tensor,
    patient_id,
    grid_size=4,
):
    channels, height, width = (
        image_tensor.shape
    )

    if (
        height % grid_size != 0
        or width % grid_size != 0
    ):
        raise ValueError(
            "Image dimensions must be divisible "
            "by the patch grid size."
        )

    patch_height = height // grid_size
    patch_width = width // grid_size

    patches = (
        image_tensor
        .reshape(
            channels,
            grid_size,
            patch_height,
            grid_size,
            patch_width,
        )
        .permute(
            1,
            3,
            0,
            2,
            4,
        )
        .reshape(
            grid_size * grid_size,
            channels,
            patch_height,
            patch_width,
        )
    )

    generator = torch.Generator()

    generator.manual_seed(
        stable_patient_seed(
            patient_id
        )
    )

    permutation = torch.randperm(
        grid_size * grid_size,
        generator=generator,
    )

    shuffled_patches = patches[
        permutation
    ]

    shuffled_tensor = (
        shuffled_patches
        .reshape(
            grid_size,
            grid_size,
            channels,
            patch_height,
            patch_width,
        )
        .permute(
            2,
            0,
            3,
            1,
            4,
        )
        .reshape(
            channels,
            height,
            width,
        )
    )

    return shuffled_tensor


# ------------------------------------------------------------
# 3. Apply one pre-registered perturbation
# ------------------------------------------------------------

def perturb_and_normalise(
    image,
    condition,
    patient_id,
):
    image = prepare_clean_crop(
        image
    )

    if condition == "grayscale":
        image = TF.rgb_to_grayscale(
            image,
            num_output_channels=3,
        )

        tensor = to_tensor_transform(
            image
        )

    elif condition == "strong_blur":
        image = TF.gaussian_blur(
            image,
            kernel_size=[31, 31],
            sigma=[10.0, 10.0],
        )

        tensor = to_tensor_transform(
            image
        )

    elif condition == "patch_shuffle_4x4":
        tensor = to_tensor_transform(
            image
        )

        tensor = shuffle_tensor_patches(
            image_tensor=tensor,
            patient_id=patient_id,
            grid_size=4,
        )

    else:
        raise ValueError(
            f"Unknown perturbation condition: "
            f"{condition}"
        )

    tensor = normalise_transform(
        tensor
    )

    return tensor


# ------------------------------------------------------------
# 4. Dataset
# ------------------------------------------------------------

class PerturbedDeepDRiDDataset(Dataset):
    def __init__(
        self,
        metadata,
        condition,
    ):
        self.image_paths = (
            metadata["image_path_disk"]
            .astype(str)
            .tolist()
        )

        self.patient_ids = (
            metadata["patient_id"]
            .astype(str)
            .tolist()
        )

        self.condition = condition

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        image_path = self.image_paths[
            index
        ]

        patient_id = self.patient_ids[
            index
        ]

        try:
            with Image.open(
                image_path
            ) as image:
                image = image.convert("RGB")

                tensor = perturb_and_normalise(
                    image=image,
                    condition=self.condition,
                    patient_id=patient_id,
                )

        except Exception as error:
            raise RuntimeError(
                f"Failed condition={self.condition}, "
                f"row={index}, path={image_path}"
            ) from error

        return tensor, index


# ------------------------------------------------------------
# 5. Extract each condition
# ------------------------------------------------------------

PERTURBATION_CONDITIONS = [
    "grayscale",
    "strong_blur",
    "patch_shuffle_4x4",
]

BATCH_SIZE = 32
NUM_WORKERS = 2

condition_embeddings = {
    "clean_cached": clean_image_embeddings
}

embedding_audit_rows = []

device_type = device.type

encoder.eval()

for condition in PERTURBATION_CONDITIONS:
    output_path = (
        KILL_TEST_ROOT
        / (
            "ResNet50_ImageNet1K_V2_"
            f"{condition}_ImageEmbeddings_float32.npy"
        )
    )

    if output_path.is_file():
        print(
            f"\nLoading cached condition: "
            f"{condition}"
        )

        condition_matrix = np.load(
            output_path
        )

    else:
        print(
            f"\nExtracting condition: "
            f"{condition}"
        )

        dataset = PerturbedDeepDRiDDataset(
            metadata=image_meta,
            condition=condition,
        )

        loader = DataLoader(
            dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=True,
            persistent_workers=(
                NUM_WORKERS > 0
            ),
        )

        condition_matrix = np.empty(
            (
                len(image_meta),
                embedding_dim,
            ),
            dtype=np.float32,
        )

        with torch.inference_mode():
            for (
                image_batch,
                row_indices,
            ) in tqdm(
                loader,
                desc=condition,
            ):
                image_batch = image_batch.to(
                    device,
                    non_blocking=True,
                )

                with torch.autocast(
                    device_type=device_type,
                    dtype=torch.float16,
                    enabled=True,
                ):
                    batch_embeddings = encoder(
                        image_batch
                    )

                batch_embeddings = (
                    batch_embeddings
                    .float()
                    .cpu()
                    .numpy()
                )

                row_indices = (
                    row_indices
                    .numpy()
                )

                condition_matrix[
                    row_indices
                ] = batch_embeddings

        np.save(
            output_path,
            condition_matrix,
            allow_pickle=False,
        )

        print("Saved:")
        print(output_path)

    assert condition_matrix.shape == (
        1600,
        embedding_dim,
    )

    assert (
        condition_matrix.dtype
        == np.float32
    )

    assert np.isfinite(
        condition_matrix
    ).all()

    condition_embeddings[
        condition
    ] = condition_matrix

    norms = np.linalg.norm(
        condition_matrix,
        axis=1,
    )

    dimension_sd = (
        condition_matrix.std(
            axis=0
        )
    )

    embedding_audit_rows.append(
        {
            "condition": condition,
            "rows": len(
                condition_matrix
            ),
            "dimensions": (
                condition_matrix.shape[1]
            ),
            "norm_min": float(
                norms.min()
            ),
            "norm_mean": float(
                norms.mean()
            ),
            "norm_max": float(
                norms.max()
            ),
            "near_zero_variance_dimensions": int(
                (
                    dimension_sd
                    < 1e-8
                ).sum()
            ),
            "output_path": str(
                output_path
            ),
        }
    )


# ------------------------------------------------------------
# 6. Save and report embedding audits
# ------------------------------------------------------------

embedding_audit = pd.DataFrame(
    embedding_audit_rows
)

EMBEDDING_AUDIT_PATH = (
    KILL_TEST_ROOT
    / "Perturbation_Embedding_Audit.csv"
)

embedding_audit.to_csv(
    EMBEDDING_AUDIT_PATH,
    index=False,
)

print(
    "\n================ PERTURBATION "
    "EMBEDDING AUDIT ================"
)

display(
    embedding_audit[
        [
            "condition",
            "rows",
            "dimensions",
            "norm_min",
            "norm_mean",
            "norm_max",
            "near_zero_variance_dimensions",
        ]
    ]
)

print("\nSaved audit:")
print(EMBEDDING_AUDIT_PATH)

print(
    "\nAll pre-registered perturbation "
    "embeddings passed integrity checks."
)


Extracting condition: grayscale


grayscale:   0%|          | 0/50 [00:00<?, ?it/s]

Saved:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Locality_Kill_Test_v0.1/ResNet50_ImageNet1K_V2_grayscale_ImageEmbeddings_float32.npy

Extracting condition: strong_blur


strong_blur:   0%|          | 0/50 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b4b31a034c0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b4b31a034c0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

Saved:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Locality_Kill_Test_v0.1/ResNet50_ImageNet1K_V2_strong_blur_ImageEmbeddings_float32.npy

Extracting condition: patch_shuffle_4x4


patch_shuffle_4x4:   0%|          | 0/50 [00:10<?, ?it/s]

Saved:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Locality_Kill_Test_v0.1/ResNet50_ImageNet1K_V2_patch_shuffle_4x4_ImageEmbeddings_float32.npy

================ PERTURBATION EMBEDDING AUDIT ================


,condition,rows,dimensions,norm_min,norm_mean,norm_max,near_zero_variance_dimensions
0,grayscale,1600,2048,8.232087,13.111636,16.498518,1
1,strong_blur,1600,2048,11.331522,13.747149,16.958448,10
2,patch_shuffle_4x4,1600,2048,10.567918,16.472807,22.071596,0



Saved audit:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Locality_Kill_Test_v0.1/Perturbation_Embedding_Audit.csv

All pre-registered perturbation embeddings passed integrity checks.


In [3]:
#@title 03. Unified paired-difference probes across kill-test conditions

from pathlib import Path
from scipy.stats import binomtest

import json
import joblib
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


RANDOM_SEED = 20260719

PAIR_C_GRID = [
    1e-5,
    3e-5,
    1e-4,
    3e-4,
    1e-3,
    3e-3,
    1e-2,
    3e-2,
    1e-1,
]


# ------------------------------------------------------------
# 1. Load all condition matrices from their sealed caches
# ------------------------------------------------------------

CONDITION_PATHS = {
    "clean_cached": (
        BASELINE_ROOT
        / "ResNet50_ImageNet1K_V2_Embeddings_float32.npy"
    ),
    "grayscale": (
        KILL_TEST_ROOT
        / "ResNet50_ImageNet1K_V2_grayscale_ImageEmbeddings_float32.npy"
    ),
    "strong_blur": (
        KILL_TEST_ROOT
        / "ResNet50_ImageNet1K_V2_strong_blur_ImageEmbeddings_float32.npy"
    ),
    "patch_shuffle_4x4": (
        KILL_TEST_ROOT
        / "ResNet50_ImageNet1K_V2_patch_shuffle_4x4_ImageEmbeddings_float32.npy"
    ),
}

condition_matrices = {}

for condition, matrix_path in CONDITION_PATHS.items():
    assert matrix_path.is_file(), matrix_path

    matrix = np.load(matrix_path)

    assert matrix.shape == (1600, 2048)
    assert matrix.dtype == np.float32
    assert np.isfinite(matrix).all()

    condition_matrices[condition] = matrix

print("Loaded conditions:")
for condition, matrix in condition_matrices.items():
    print(f"  {condition}: {matrix.shape}")


# ------------------------------------------------------------
# 2. Freeze image row alignment
# ------------------------------------------------------------

image_meta_probe = (
    image_meta
    .sort_values("embedding_row")
    .reset_index(drop=True)
    .copy()
)

image_meta_probe["patient_id"] = (
    image_meta_probe["patient_id"]
    .astype(str)
)

expected_embedding_rows = np.arange(
    len(image_meta_probe)
)

assert np.array_equal(
    image_meta_probe["embedding_row"].to_numpy(),
    expected_embedding_rows,
), "Image metadata is not aligned with embedding rows."


def canonical_eye_name(value):
    value = str(value).strip().lower()

    if value.startswith("l"):
        return "L"

    if value.startswith("r"):
        return "R"

    raise ValueError(
        f"Unrecognised eye label: {value}"
    )


image_meta_probe["canonical_eye"] = (
    image_meta_probe["eye_resolved"]
    .map(canonical_eye_name)
)


# ------------------------------------------------------------
# 3. Build a fixed eye index
#
# Each eye must have two views and one unique DR grade.
# ------------------------------------------------------------

eye_definition_records = []

for (
    split_name,
    patient_id,
    eye_name,
), row_indices in image_meta_probe.groupby(
    [
        "split",
        "patient_id",
        "canonical_eye",
    ],
    sort=False,
).groups.items():

    row_indices = np.asarray(
        list(row_indices),
        dtype=int,
    )

    group_metadata = image_meta_probe.iloc[
        row_indices
    ]

    eye_grades = (
        group_metadata["eye_DR_Level"]
        .dropna()
        .astype(int)
        .unique()
    )

    if len(row_indices) != 2:
        raise ValueError(
            f"{split_name}, patient={patient_id}, "
            f"eye={eye_name}: expected two views, "
            f"found {len(row_indices)}."
        )

    if len(eye_grades) != 1:
        raise ValueError(
            f"{split_name}, patient={patient_id}, "
            f"eye={eye_name}: inconsistent grades "
            f"{eye_grades}."
        )

    eye_definition_records.append(
        {
            "split": split_name,
            "patient_id": str(patient_id),
            "canonical_eye": eye_name,
            "eye_DR_Level": int(eye_grades[0]),
            "image_row_1": int(row_indices[0]),
            "image_row_2": int(row_indices[1]),
        }
    )


eye_definition = pd.DataFrame(
    eye_definition_records
)

eye_definition["eye_row"] = np.arange(
    len(eye_definition)
)

assert len(eye_definition) == 800

assert (
    eye_definition
    .groupby(["split", "patient_id"])
    .size()
    .eq(2)
    .all()
)


# ------------------------------------------------------------
# 4. Freeze one paired record per unequal-grade patient
# ------------------------------------------------------------

pair_definition_records = []

for (
    split_name,
    patient_id,
), patient_group in eye_definition.groupby(
    ["split", "patient_id"],
    sort=False,
):
    left_eye = patient_group.loc[
        patient_group["canonical_eye"].eq("L")
    ]

    right_eye = patient_group.loc[
        patient_group["canonical_eye"].eq("R")
    ]

    if len(left_eye) != 1 or len(right_eye) != 1:
        raise ValueError(
            f"Could not uniquely resolve both eyes for "
            f"{split_name}, patient={patient_id}."
        )

    left_eye = left_eye.iloc[0]
    right_eye = right_eye.iloc[0]

    left_grade = int(
        left_eye["eye_DR_Level"]
    )

    right_grade = int(
        right_eye["eye_DR_Level"]
    )

    if left_grade == right_grade:
        continue

    pair_definition_records.append(
        {
            "split": split_name,
            "patient_id": str(patient_id),
            "left_eye_row": int(
                left_eye["eye_row"]
            ),
            "right_eye_row": int(
                right_eye["eye_row"]
            ),
            "left_grade": left_grade,
            "right_grade": right_grade,
            "left_higher_grade": int(
                left_grade > right_grade
            ),
            "absolute_grade_gap": abs(
                left_grade - right_grade
            ),
            "binary_referable_discordant": int(
                (left_grade >= 2)
                != (right_grade >= 2)
            ),
        }
    )


pair_definition = pd.DataFrame(
    pair_definition_records
)

training_pair_mask = (
    pair_definition["split"]
    .eq("training")
    .to_numpy()
)

validation_pair_mask = (
    pair_definition["split"]
    .eq("validation")
    .to_numpy()
)

training_pair_definition = (
    pair_definition.loc[
        training_pair_mask
    ]
    .reset_index(drop=True)
    .copy()
)

validation_pair_definition = (
    pair_definition.loc[
        validation_pair_mask
    ]
    .reset_index(drop=True)
    .copy()
)

assert len(training_pair_definition) == 134
assert len(validation_pair_definition) == 45

print(
    "\nTraining unequal-grade patients:",
    len(training_pair_definition),
)

print(
    "Validation unequal-grade patients:",
    len(validation_pair_definition),
)


# ------------------------------------------------------------
# 5. Shared model helpers
# ------------------------------------------------------------

def symmetrically_augment(
    feature_matrix,
    labels,
):
    return (
        np.concatenate(
            [
                feature_matrix,
                -feature_matrix,
            ],
            axis=0,
        ),
        np.concatenate(
            [
                labels,
                1 - labels,
            ],
            axis=0,
        ),
    )


def make_paired_probe(C_value):
    return Pipeline(
        steps=[
            (
                "standardize",
                StandardScaler(
                    with_mean=False
                ),
            ),
            (
                "classifier",
                LogisticRegression(
                    penalty="l2",
                    C=C_value,
                    solver="liblinear",
                    fit_intercept=False,
                    max_iter=10000,
                    random_state=RANDOM_SEED,
                ),
            ),
        ]
    )


def build_eye_embeddings(
    image_embedding_matrix,
):
    eye_embeddings = np.empty(
        (
            len(eye_definition),
            image_embedding_matrix.shape[1],
        ),
        dtype=np.float32,
    )

    for row in eye_definition.itertuples(
        index=False
    ):
        eye_embeddings[row.eye_row] = (
            image_embedding_matrix[
                [
                    row.image_row_1,
                    row.image_row_2,
                ]
            ]
            .mean(axis=0)
        )

    assert np.isfinite(
        eye_embeddings
    ).all()

    return eye_embeddings


def build_paired_features(
    eye_embeddings,
):
    left_rows = (
        pair_definition["left_eye_row"]
        .to_numpy(dtype=int)
    )

    right_rows = (
        pair_definition["right_eye_row"]
        .to_numpy(dtype=int)
    )

    paired_features = (
        eye_embeddings[left_rows]
        - eye_embeddings[right_rows]
    ).astype(np.float32)

    paired_labels = (
        pair_definition["left_higher_grade"]
        .to_numpy(dtype=int)
    )

    return paired_features, paired_labels


# ------------------------------------------------------------
# 6. Training-only hyperparameter selection
# ------------------------------------------------------------

def select_paired_probe_C(
    X_train,
    y_train,
):
    cv_rows = []

    for C_value in PAIR_C_GRID:
        fold_accuracy = []
        fold_balanced_accuracy = []
        fold_auc = []

        splitter = StratifiedKFold(
            n_splits=5,
            shuffle=True,
            random_state=RANDOM_SEED,
        )

        for (
            fold_train_indices,
            fold_test_indices,
        ) in splitter.split(
            X_train,
            y_train,
        ):
            (
                augmented_features,
                augmented_labels,
            ) = symmetrically_augment(
                X_train[
                    fold_train_indices
                ],
                y_train[
                    fold_train_indices
                ],
            )

            model = make_paired_probe(
                C_value
            )

            model.fit(
                augmented_features,
                augmented_labels,
            )

            probabilities = (
                model.predict_proba(
                    X_train[
                        fold_test_indices
                    ]
                )[:, 1]
            )

            predictions = (
                probabilities >= 0.5
            ).astype(int)

            fold_labels = y_train[
                fold_test_indices
            ]

            fold_accuracy.append(
                accuracy_score(
                    fold_labels,
                    predictions,
                )
            )

            fold_balanced_accuracy.append(
                balanced_accuracy_score(
                    fold_labels,
                    predictions,
                )
            )

            fold_auc.append(
                roc_auc_score(
                    fold_labels,
                    probabilities,
                )
            )

        cv_rows.append(
            {
                "C": C_value,
                "mean_cv_accuracy": float(
                    np.mean(fold_accuracy)
                ),
                "sd_cv_accuracy": float(
                    np.std(
                        fold_accuracy,
                        ddof=1,
                    )
                ),
                "mean_cv_balanced_accuracy": float(
                    np.mean(
                        fold_balanced_accuracy
                    )
                ),
                "sd_cv_balanced_accuracy": float(
                    np.std(
                        fold_balanced_accuracy,
                        ddof=1,
                    )
                ),
                "mean_cv_auc": float(
                    np.mean(fold_auc)
                ),
            }
        )

    cv_results = pd.DataFrame(
        cv_rows
    )

    cv_results = (
        cv_results
        .sort_values(
            by=[
                "mean_cv_balanced_accuracy",
                "mean_cv_accuracy",
                "C",
            ],
            ascending=[
                False,
                False,
                True,
            ],
        )
        .reset_index(drop=True)
    )

    return (
        float(cv_results.loc[0, "C"]),
        cv_results,
    )


# ------------------------------------------------------------
# 7. Fit and evaluate one condition
# ------------------------------------------------------------

def evaluate_condition(
    condition_name,
    image_embedding_matrix,
):
    eye_embeddings = build_eye_embeddings(
        image_embedding_matrix
    )

    paired_features, paired_labels = (
        build_paired_features(
            eye_embeddings
        )
    )

    X_train = paired_features[
        training_pair_mask
    ]

    X_validation = paired_features[
        validation_pair_mask
    ]

    y_train = paired_labels[
        training_pair_mask
    ]

    y_validation = paired_labels[
        validation_pair_mask
    ]

    best_C, cv_results = (
        select_paired_probe_C(
            X_train,
            y_train,
        )
    )

    (
        augmented_training_features,
        augmented_training_labels,
    ) = symmetrically_augment(
        X_train,
        y_train,
    )

    final_model = make_paired_probe(
        best_C
    )

    final_model.fit(
        augmented_training_features,
        augmented_training_labels,
    )

    validation_probability = (
        final_model.predict_proba(
            X_validation
        )[:, 1]
    )

    validation_prediction = (
        validation_probability >= 0.5
    ).astype(int)

    validation_decision = (
        final_model.decision_function(
            X_validation
        )
    )

    validation_correct = (
        validation_prediction
        == y_validation
    ).astype(int)

    validation_accuracy = accuracy_score(
        y_validation,
        validation_prediction,
    )

    validation_balanced_accuracy = (
        balanced_accuracy_score(
            y_validation,
            validation_prediction,
        )
    )

    validation_auc = roc_auc_score(
        y_validation,
        validation_probability,
    )

    correct_pairs = int(
        validation_correct.sum()
    )

    exact_test = binomtest(
        k=correct_pairs,
        n=len(y_validation),
        p=0.5,
        alternative="greater",
    )

    exact_ci = binomtest(
        k=correct_pairs,
        n=len(y_validation),
    ).proportion_ci(
        confidence_level=0.95,
        method="exact",
    )

    true_direction_sign = (
        2 * y_validation - 1
    )

    true_signed_margin = (
        validation_decision
        * true_direction_sign
    )

    prediction_table = (
        validation_pair_definition
        .copy()
    )

    prediction_table[
        "condition"
    ] = condition_name

    prediction_table[
        "probability_left_higher"
    ] = validation_probability

    prediction_table[
        "predicted_left_higher"
    ] = validation_prediction

    prediction_table[
        "correct"
    ] = validation_correct

    prediction_table[
        "true_signed_margin"
    ] = true_signed_margin

    summary = {
        "condition": condition_name,
        "selected_C": best_C,
        "training_best_mean_cv_balanced_accuracy": float(
            cv_results.loc[
                0,
                "mean_cv_balanced_accuracy",
            ]
        ),
        "validation_patients": int(
            len(y_validation)
        ),
        "correct_pairs": correct_pairs,
        "validation_accuracy": float(
            validation_accuracy
        ),
        "accuracy_ci_lower": float(
            exact_ci.low
        ),
        "accuracy_ci_upper": float(
            exact_ci.high
        ),
        "validation_balanced_accuracy": float(
            validation_balanced_accuracy
        ),
        "validation_directional_auc": float(
            validation_auc
        ),
        "exact_one_sided_p": float(
            exact_test.pvalue
        ),
        "mean_true_signed_margin": float(
            true_signed_margin.mean()
        ),
    }

    return {
        "summary": summary,
        "cv_results": cv_results,
        "predictions": prediction_table,
        "model": final_model,
    }


# ------------------------------------------------------------
# 8. Evaluate clean and all perturbations
# ------------------------------------------------------------

condition_results = {}

for (
    condition_name,
    image_embedding_matrix,
) in condition_matrices.items():

    print(
        f"\nEvaluating: {condition_name}"
    )

    result = evaluate_condition(
        condition_name=condition_name,
        image_embedding_matrix=(
            image_embedding_matrix
        ),
    )

    condition_results[
        condition_name
    ] = result

    print(
        "  Selected C:",
        result["summary"][
            "selected_C"
        ],
    )

    print(
        "  Accuracy:",
        f"{result['summary']['validation_accuracy']:.4f}",
    )

    print(
        "  Directional AUC:",
        f"{result['summary']['validation_directional_auc']:.4f}",
    )


# ------------------------------------------------------------
# 9. Verify exact clean-baseline reproduction
# ------------------------------------------------------------

clean_summary = condition_results[
    "clean_cached"
]["summary"]

clean_predictions = condition_results[
    "clean_cached"
]["predictions"]

baseline_accuracy = float(
    baseline_decision[
        "primary_locality_result"
    ]["accuracy"]
)

assert np.isclose(
    clean_summary[
        "validation_accuracy"
    ],
    baseline_accuracy,
)

assert (
    clean_summary["correct_pairs"]
    == 31
)

baseline_correctness = (
    baseline_pair_predictions[
        ["patient_id", "correct"]
    ]
    .copy()
)

baseline_correctness["patient_id"] = (
    baseline_correctness["patient_id"]
    .astype(str)
)

clean_correctness = (
    clean_predictions[
        ["patient_id", "correct"]
    ]
    .copy()
)

clean_reproduction_check = (
    baseline_correctness
    .merge(
        clean_correctness,
        on="patient_id",
        suffixes=(
            "_baseline",
            "_recomputed",
        ),
        validate="one_to_one",
    )
)

clean_correctness_agreement = float(
    (
        clean_reproduction_check[
            "correct_baseline"
        ].astype(int)
        ==
        clean_reproduction_check[
            "correct_recomputed"
        ].astype(int)
    ).mean()
)

assert clean_correctness_agreement == 1.0

print(
    "\nClean baseline correctness agreement:",
    clean_correctness_agreement,
)


# ------------------------------------------------------------
# 10. Paired comparisons against clean on the same 45 patients
# ------------------------------------------------------------

def bootstrap_accuracy_difference(
    clean_correct,
    condition_correct,
    number_of_bootstraps=20000,
    random_seed=RANDOM_SEED,
):
    clean_correct = np.asarray(
        clean_correct,
        dtype=float,
    )

    condition_correct = np.asarray(
        condition_correct,
        dtype=float,
    )

    differences = (
        condition_correct
        - clean_correct
    )

    rng = np.random.default_rng(
        random_seed
    )

    bootstrap_values = np.empty(
        number_of_bootstraps,
        dtype=float,
    )

    for bootstrap_index in range(
        number_of_bootstraps
    ):
        sampled_indices = rng.integers(
            low=0,
            high=len(differences),
            size=len(differences),
        )

        bootstrap_values[
            bootstrap_index
        ] = differences[
            sampled_indices
        ].mean()

    return {
        "estimate": float(
            differences.mean()
        ),
        "lower": float(
            np.percentile(
                bootstrap_values,
                2.5,
            )
        ),
        "upper": float(
            np.percentile(
                bootstrap_values,
                97.5,
            )
        ),
    }


clean_prediction_table = (
    condition_results[
        "clean_cached"
    ]["predictions"]
    .sort_values("patient_id")
    .reset_index(drop=True)
)

paired_comparison_rows = []

for comparison_index, condition_name in enumerate(
    [
        "grayscale",
        "strong_blur",
        "patch_shuffle_4x4",
    ]
):
    condition_prediction_table = (
        condition_results[
            condition_name
        ]["predictions"]
        .sort_values("patient_id")
        .reset_index(drop=True)
    )

    assert np.array_equal(
        clean_prediction_table[
            "patient_id"
        ].to_numpy(),
        condition_prediction_table[
            "patient_id"
        ].to_numpy(),
    )

    clean_correct = (
        clean_prediction_table[
            "correct"
        ].to_numpy(dtype=int)
    )

    condition_correct = (
        condition_prediction_table[
            "correct"
        ].to_numpy(dtype=int)
    )

    difference_result = (
        bootstrap_accuracy_difference(
            clean_correct=clean_correct,
            condition_correct=(
                condition_correct
            ),
            number_of_bootstraps=20000,
            random_seed=(
                RANDOM_SEED
                + 100
                + comparison_index
            ),
        )
    )

    clean_only_correct = int(
        (
            (clean_correct == 1)
            & (condition_correct == 0)
        ).sum()
    )

    condition_only_correct = int(
        (
            (clean_correct == 0)
            & (condition_correct == 1)
        ).sum()
    )

    discordant_outcomes = (
        clean_only_correct
        + condition_only_correct
    )

    if discordant_outcomes > 0:
        exact_mcnemar_p = float(
            binomtest(
                k=condition_only_correct,
                n=discordant_outcomes,
                p=0.5,
                alternative="two-sided",
            ).pvalue
        )

    else:
        exact_mcnemar_p = 1.0

    condition_summary = (
        condition_results[
            condition_name
        ]["summary"]
    )

    paired_comparison_rows.append(
        {
            "condition": condition_name,
            "clean_accuracy": clean_summary[
                "validation_accuracy"
            ],
            "condition_accuracy": condition_summary[
                "validation_accuracy"
            ],
            "condition_minus_clean_accuracy": (
                difference_result[
                    "estimate"
                ]
            ),
            "difference_ci_lower": (
                difference_result[
                    "lower"
                ]
            ),
            "difference_ci_upper": (
                difference_result[
                    "upper"
                ]
            ),
            "clean_only_correct": (
                clean_only_correct
            ),
            "condition_only_correct": (
                condition_only_correct
            ),
            "exact_mcnemar_p": (
                exact_mcnemar_p
            ),
        }
    )


paired_comparisons = pd.DataFrame(
    paired_comparison_rows
)


# ------------------------------------------------------------
# 11. Consolidate and save
# ------------------------------------------------------------

condition_summary_table = pd.DataFrame(
    [
        result["summary"]
        for result in condition_results.values()
    ]
)

all_prediction_tables = pd.concat(
    [
        result["predictions"]
        for result in condition_results.values()
    ],
    ignore_index=True,
)

SUMMARY_TABLE_PATH = (
    KILL_TEST_ROOT
    / "Kill_Test_PairedDifference_Condition_Summary.csv"
)

PAIRED_COMPARISON_PATH = (
    KILL_TEST_ROOT
    / "Kill_Test_Paired_Comparisons_vs_Clean.csv"
)

ALL_PREDICTIONS_PATH = (
    KILL_TEST_ROOT
    / "Kill_Test_All_Validation_Pair_Predictions.csv"
)

condition_summary_table.to_csv(
    SUMMARY_TABLE_PATH,
    index=False,
)

paired_comparisons.to_csv(
    PAIRED_COMPARISON_PATH,
    index=False,
)

all_prediction_tables.to_csv(
    ALL_PREDICTIONS_PATH,
    index=False,
)

for condition_name, result in condition_results.items():
    result["cv_results"].to_csv(
        KILL_TEST_ROOT
        / f"{condition_name}_PairedProbe_CV.csv",
        index=False,
    )

    joblib.dump(
        result["model"],
        KILL_TEST_ROOT
        / f"{condition_name}_PairedProbe.joblib",
    )


# ------------------------------------------------------------
# 12. Final report
# ------------------------------------------------------------

print(
    "\n================ KILL-TEST CONDITION "
    "RESULTS ================"
)

display(
    condition_summary_table[
        [
            "condition",
            "selected_C",
            "correct_pairs",
            "validation_accuracy",
            "accuracy_ci_lower",
            "accuracy_ci_upper",
            "validation_balanced_accuracy",
            "validation_directional_auc",
            "exact_one_sided_p",
        ]
    ]
)

print(
    "\n================ PAIRED CHANGES "
    "VERSUS CLEAN ================"
)

display(paired_comparisons)

print("\nSaved:")
print(SUMMARY_TABLE_PATH)
print(PAIRED_COMPARISON_PATH)
print(ALL_PREDICTIONS_PATH)

print(
    "\nUnified locality kill-test probes passed."
)

Loaded conditions:
  clean_cached: (1600, 2048)
  grayscale: (1600, 2048)
  strong_blur: (1600, 2048)
  patch_shuffle_4x4: (1600, 2048)

Training unequal-grade patients: 134
Validation unequal-grade patients: 45

Evaluating: clean_cached
  Selected C: 0.001
  Accuracy: 0.6889
  Directional AUC: 0.7530

Evaluating: grayscale
  Selected C: 0.01
  Accuracy: 0.6889
  Directional AUC: 0.7308

Evaluating: strong_blur
  Selected C: 0.1
  Accuracy: 0.5333
  Directional AUC: 0.5101

Evaluating: patch_shuffle_4x4
  Selected C: 0.0003
  Accuracy: 0.7111
  Directional AUC: 0.7389

Clean baseline correctness agreement: 1.0

================ KILL-TEST CONDITION RESULTS ================


,condition,selected_C,correct_pairs,validation_accuracy,accuracy_ci_lower,accuracy_ci_upper,validation_balanced_accuracy,validation_directional_auc,exact_one_sided_p
0,clean_cached,0.0010,31,0.688889,0.533509,0.818341,0.688259,0.753036,0.008047
1,grayscale,0.0100,31,0.688889,0.533509,0.818341,0.702429,0.730769,0.008047
2,strong_blur,0.1000,24,0.533333,0.378720,0.683399,0.546559,0.510121,0.382996
3,patch_shuffle_4x4,0.0003,32,0.711111,0.556855,0.836337,0.721660,0.738866,0.003304



================ PAIRED CHANGES VERSUS CLEAN ================


,condition,clean_accuracy,condition_accuracy,condition_minus_clean_accuracy,difference_ci_lower,difference_ci_upper,clean_only_correct,condition_only_correct,exact_mcnemar_p
0,grayscale,0.688889,0.688889,0.000000,-0.177778,0.177778,8,8,1.000000
1,strong_blur,0.688889,0.533333,-0.155556,-0.333333,0.022222,13,6,0.167068
2,patch_shuffle_4x4,0.688889,0.711111,0.022222,-0.111111,0.155556,4,5,1.000000



Saved:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Locality_Kill_Test_v0.1/Kill_Test_PairedDifference_Condition_Summary.csv
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Locality_Kill_Test_v0.1/Kill_Test_Paired_Comparisons_vs_Clean.csv
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Locality_Kill_Test_v0.1/Kill_Test_All_Validation_Pair_Predictions.csv

Unified locality kill-test probes passed.


In [4]:
#@title 04. Fixed clean-probe transfer across perturbations

from scipy.stats import binomtest

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Recover the model trained only on clean embeddings
# ------------------------------------------------------------

assert "condition_results" in globals()
assert "condition_matrices" in globals()

clean_fixed_probe = condition_results[
    "clean_cached"
]["model"]

clean_selected_C = condition_results[
    "clean_cached"
]["summary"]["selected_C"]

print("Frozen clean-probe C:", clean_selected_C)


# ------------------------------------------------------------
# 2. Bootstrap paired accuracy change
# ------------------------------------------------------------

def paired_accuracy_difference_bootstrap(
    reference_correct,
    condition_correct,
    number_of_bootstraps=20000,
    random_seed=20260719,
):
    reference_correct = np.asarray(
        reference_correct,
        dtype=float,
    )

    condition_correct = np.asarray(
        condition_correct,
        dtype=float,
    )

    patient_level_difference = (
        condition_correct
        - reference_correct
    )

    rng = np.random.default_rng(
        random_seed
    )

    bootstrap_estimates = np.empty(
        number_of_bootstraps,
        dtype=float,
    )

    for bootstrap_index in range(
        number_of_bootstraps
    ):
        sampled_indices = rng.integers(
            low=0,
            high=len(patient_level_difference),
            size=len(patient_level_difference),
        )

        bootstrap_estimates[
            bootstrap_index
        ] = patient_level_difference[
            sampled_indices
        ].mean()

    return {
        "estimate": float(
            patient_level_difference.mean()
        ),
        "lower": float(
            np.percentile(
                bootstrap_estimates,
                2.5,
            )
        ),
        "upper": float(
            np.percentile(
                bootstrap_estimates,
                97.5,
            )
        ),
    }


# ------------------------------------------------------------
# 3. Apply the unchanged clean probe to every condition
# ------------------------------------------------------------

fixed_probe_summary_rows = []
fixed_probe_prediction_tables = []

validation_labels_reference = None
clean_fixed_correct = None

for condition_index, (
    condition_name,
    image_embedding_matrix,
) in enumerate(
    condition_matrices.items()
):
    # Use the same eye averaging and left-minus-right
    # construction as the sealed baseline.
    condition_eye_embeddings = (
        build_eye_embeddings(
            image_embedding_matrix
        )
    )

    (
        condition_pair_features,
        condition_pair_labels,
    ) = build_paired_features(
        condition_eye_embeddings
    )

    X_validation_condition = (
        condition_pair_features[
            validation_pair_mask
        ]
    )

    y_validation_condition = (
        condition_pair_labels[
            validation_pair_mask
        ]
    )

    if validation_labels_reference is None:
        validation_labels_reference = (
            y_validation_condition.copy()
        )

    assert np.array_equal(
        y_validation_condition,
        validation_labels_reference,
    )

    # Crucially, the model and scaler are not refitted.
    condition_probability = (
        clean_fixed_probe.predict_proba(
            X_validation_condition
        )[:, 1]
    )

    condition_decision = (
        clean_fixed_probe.decision_function(
            X_validation_condition
        )
    )

    condition_prediction = (
        condition_probability >= 0.5
    ).astype(int)

    condition_correct = (
        condition_prediction
        == y_validation_condition
    ).astype(int)

    correct_pairs = int(
        condition_correct.sum()
    )

    number_of_pairs = int(
        len(condition_correct)
    )

    condition_accuracy = float(
        condition_correct.mean()
    )

    condition_balanced_accuracy = float(
        balanced_accuracy_score(
            y_validation_condition,
            condition_prediction,
        )
    )

    condition_auc = float(
        roc_auc_score(
            y_validation_condition,
            condition_probability,
        )
    )

    condition_binomial_test = binomtest(
        k=correct_pairs,
        n=number_of_pairs,
        p=0.5,
        alternative="greater",
    )

    condition_exact_ci = binomtest(
        k=correct_pairs,
        n=number_of_pairs,
    ).proportion_ci(
        confidence_level=0.95,
        method="exact",
    )

    true_direction_sign = (
        2 * y_validation_condition - 1
    )

    true_signed_margin = (
        condition_decision
        * true_direction_sign
    )

    prediction_table = (
        validation_pair_definition.copy()
    )

    prediction_table["condition"] = (
        condition_name
    )

    prediction_table["probability_left_higher"] = (
        condition_probability
    )

    prediction_table["predicted_left_higher"] = (
        condition_prediction
    )

    prediction_table["correct"] = (
        condition_correct
    )

    prediction_table["true_signed_margin"] = (
        true_signed_margin
    )

    fixed_probe_prediction_tables.append(
        prediction_table
    )

    if condition_name == "clean_cached":
        clean_fixed_correct = (
            condition_correct.copy()
        )

        accuracy_difference = {
            "estimate": 0.0,
            "lower": 0.0,
            "upper": 0.0,
        }

        clean_only_correct = 0
        condition_only_correct = 0
        exact_mcnemar_p = 1.0

    else:
        accuracy_difference = (
            paired_accuracy_difference_bootstrap(
                reference_correct=(
                    clean_fixed_correct
                ),
                condition_correct=(
                    condition_correct
                ),
                number_of_bootstraps=20000,
                random_seed=(
                    20260719
                    + 200
                    + condition_index
                ),
            )
        )

        clean_only_correct = int(
            (
                (clean_fixed_correct == 1)
                & (condition_correct == 0)
            ).sum()
        )

        condition_only_correct = int(
            (
                (clean_fixed_correct == 0)
                & (condition_correct == 1)
            ).sum()
        )

        discordant_outcomes = (
            clean_only_correct
            + condition_only_correct
        )

        if discordant_outcomes > 0:
            exact_mcnemar_p = float(
                binomtest(
                    k=condition_only_correct,
                    n=discordant_outcomes,
                    p=0.5,
                    alternative="two-sided",
                ).pvalue
            )
        else:
            exact_mcnemar_p = 1.0

    fixed_probe_summary_rows.append(
        {
            "condition": condition_name,
            "probe_training_condition": (
                "clean_cached"
            ),
            "probe_refitted": False,
            "selected_C": float(
                clean_selected_C
            ),
            "correct_pairs": correct_pairs,
            "validation_accuracy": (
                condition_accuracy
            ),
            "accuracy_ci_lower": float(
                condition_exact_ci.low
            ),
            "accuracy_ci_upper": float(
                condition_exact_ci.high
            ),
            "validation_balanced_accuracy": (
                condition_balanced_accuracy
            ),
            "validation_directional_auc": (
                condition_auc
            ),
            "exact_one_sided_p": float(
                condition_binomial_test.pvalue
            ),
            "mean_true_signed_margin": float(
                true_signed_margin.mean()
            ),
            "condition_minus_clean_accuracy": (
                accuracy_difference["estimate"]
            ),
            "difference_ci_lower": (
                accuracy_difference["lower"]
            ),
            "difference_ci_upper": (
                accuracy_difference["upper"]
            ),
            "clean_only_correct": (
                clean_only_correct
            ),
            "condition_only_correct": (
                condition_only_correct
            ),
            "exact_mcnemar_p": (
                exact_mcnemar_p
            ),
        }
    )


# ------------------------------------------------------------
# 4. Validate clean reproduction
# ------------------------------------------------------------

fixed_probe_summary = pd.DataFrame(
    fixed_probe_summary_rows
)

clean_fixed_row = fixed_probe_summary.loc[
    fixed_probe_summary[
        "condition"
    ].eq("clean_cached")
].iloc[0]

assert int(
    clean_fixed_row["correct_pairs"]
) == 31

assert np.isclose(
    clean_fixed_row[
        "validation_accuracy"
    ],
    31 / 45,
)

print(
    "Clean fixed-probe baseline reproduced:",
    f"{int(clean_fixed_row['correct_pairs'])}/45",
)


# ------------------------------------------------------------
# 5. Save results
# ------------------------------------------------------------

fixed_probe_predictions = pd.concat(
    fixed_probe_prediction_tables,
    ignore_index=True,
)

FIXED_PROBE_SUMMARY_PATH = (
    KILL_TEST_ROOT
    / "Kill_Test_FixedCleanProbe_Condition_Summary.csv"
)

FIXED_PROBE_PREDICTIONS_PATH = (
    KILL_TEST_ROOT
    / "Kill_Test_FixedCleanProbe_All_Predictions.csv"
)

fixed_probe_summary.to_csv(
    FIXED_PROBE_SUMMARY_PATH,
    index=False,
)

fixed_probe_predictions.to_csv(
    FIXED_PROBE_PREDICTIONS_PATH,
    index=False,
)


# ------------------------------------------------------------
# 6. Final report
# ------------------------------------------------------------

print(
    "\n================ FIXED CLEAN-PROBE "
    "TRANSFER RESULTS ================"
)

display(
    fixed_probe_summary[
        [
            "condition",
            "correct_pairs",
            "validation_accuracy",
            "accuracy_ci_lower",
            "accuracy_ci_upper",
            "validation_balanced_accuracy",
            "validation_directional_auc",
            "exact_one_sided_p",
            "condition_minus_clean_accuracy",
            "difference_ci_lower",
            "difference_ci_upper",
            "clean_only_correct",
            "condition_only_correct",
            "exact_mcnemar_p",
        ]
    ]
)

print("\nSaved:")
print(FIXED_PROBE_SUMMARY_PATH)
print(FIXED_PROBE_PREDICTIONS_PATH)

print(
    "\nFixed clean-probe transfer test passed."
)

Frozen clean-probe C: 0.001
Clean fixed-probe baseline reproduced: 31/45

================ FIXED CLEAN-PROBE TRANSFER RESULTS ================


,condition,correct_pairs,validation_accuracy,accuracy_ci_lower,accuracy_ci_upper,validation_balanced_accuracy,validation_directional_auc,exact_one_sided_p,condition_minus_clean_accuracy,difference_ci_lower,difference_ci_upper,clean_only_correct,condition_only_correct,exact_mcnemar_p
0,clean_cached,31,0.688889,0.533509,0.818341,0.688259,0.753036,0.008047,0.000000,0.000000,0.000000,0,0,1.000000
1,grayscale,31,0.688889,0.533509,0.818341,0.695344,0.811741,0.008047,0.000000,-0.155556,0.155556,6,6,1.000000
2,strong_blur,20,0.444444,0.296444,0.600027,0.420040,0.441296,0.814351,-0.244444,-0.444444,-0.044444,17,6,0.034690
3,patch_shuffle_4x4,26,0.577778,0.421503,0.723433,0.570850,0.578947,0.185649,-0.111111,-0.288889,0.066667,11,6,0.332306



Saved:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Locality_Kill_Test_v0.1/Kill_Test_FixedCleanProbe_Condition_Summary.csv
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Locality_Kill_Test_v0.1/Kill_Test_FixedCleanProbe_All_Predictions.csv

Fixed clean-probe transfer test passed.


In [5]:
#@title 05. Finalize Locality Kill Test v0.1

import json
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Retrieve rows safely
# ------------------------------------------------------------

def get_condition_row(dataframe, condition):
    matches = dataframe.loc[
        dataframe["condition"].eq(condition)
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one row for {condition}, "
            f"found {len(matches)}."
        )

    return matches.iloc[0]


adaptive_clean = get_condition_row(
    condition_summary_table,
    "clean_cached",
)

adaptive_gray = get_condition_row(
    condition_summary_table,
    "grayscale",
)

adaptive_blur = get_condition_row(
    condition_summary_table,
    "strong_blur",
)

adaptive_shuffle = get_condition_row(
    condition_summary_table,
    "patch_shuffle_4x4",
)

fixed_clean = get_condition_row(
    fixed_probe_summary,
    "clean_cached",
)

fixed_gray = get_condition_row(
    fixed_probe_summary,
    "grayscale",
)

fixed_blur = get_condition_row(
    fixed_probe_summary,
    "strong_blur",
)

fixed_shuffle = get_condition_row(
    fixed_probe_summary,
    "patch_shuffle_4x4",
)


# ------------------------------------------------------------
# 2. Mechanism gates
# ------------------------------------------------------------

clean_reproduced = bool(
    int(fixed_clean["correct_pairs"]) == 31
    and np.isclose(
        fixed_clean["validation_accuracy"],
        31 / 45,
    )
)

colour_is_required = bool(
    fixed_gray["validation_accuracy"]
    < fixed_clean["validation_accuracy"]
    and fixed_gray["exact_mcnemar_p"] < 0.05
)

fine_structure_dependency_detected = bool(
    fixed_blur["condition_minus_clean_accuracy"] < 0
    and fixed_blur["difference_ci_upper"] < 0
    and fixed_blur["exact_mcnemar_p"] < 0.05
    and adaptive_blur["validation_directional_auc"] < 0.60
)

global_layout_required_for_recoverability = bool(
    adaptive_shuffle["validation_accuracy"] <= 0.50
    or adaptive_shuffle["exact_one_sided_p"] >= 0.05
)

clean_axis_stable_after_patch_shuffle = bool(
    fixed_shuffle["validation_accuracy"] > 0.50
    and fixed_shuffle["exact_one_sided_p"] < 0.05
)


# ------------------------------------------------------------
# 3. Consolidated mechanism table
# ------------------------------------------------------------

mechanism_rows = []

for condition in [
    "clean_cached",
    "grayscale",
    "strong_blur",
    "patch_shuffle_4x4",
]:
    adaptive_row = get_condition_row(
        condition_summary_table,
        condition,
    )

    fixed_row = get_condition_row(
        fixed_probe_summary,
        condition,
    )

    mechanism_rows.append(
        {
            "condition": condition,
            "adaptive_correct_pairs": int(
                adaptive_row["correct_pairs"]
            ),
            "adaptive_accuracy": float(
                adaptive_row["validation_accuracy"]
            ),
            "adaptive_directional_auc": float(
                adaptive_row[
                    "validation_directional_auc"
                ]
            ),
            "adaptive_one_sided_p": float(
                adaptive_row["exact_one_sided_p"]
            ),
            "fixed_correct_pairs": int(
                fixed_row["correct_pairs"]
            ),
            "fixed_accuracy": float(
                fixed_row["validation_accuracy"]
            ),
            "fixed_directional_auc": float(
                fixed_row[
                    "validation_directional_auc"
                ]
            ),
            "fixed_one_sided_p": float(
                fixed_row["exact_one_sided_p"]
            ),
            "fixed_minus_clean_accuracy": float(
                fixed_row[
                    "condition_minus_clean_accuracy"
                ]
            ),
            "fixed_difference_ci_lower": float(
                fixed_row["difference_ci_lower"]
            ),
            "fixed_difference_ci_upper": float(
                fixed_row["difference_ci_upper"]
            ),
            "fixed_exact_mcnemar_p": float(
                fixed_row["exact_mcnemar_p"]
            ),
        }
    )


mechanism_table = pd.DataFrame(
    mechanism_rows
)


# ------------------------------------------------------------
# 4. Decision
# ------------------------------------------------------------

if (
    clean_reproduced
    and fine_structure_dependency_detected
    and not colour_is_required
):
    kill_test_decision = (
        "PASS_FINE_SCALE_ACHROMATIC_STRUCTURE_DEPENDENCY"
    )
else:
    kill_test_decision = (
        "INCONCLUSIVE_LOCALITY_MECHANISM"
    )


decision = {
    "decision": kill_test_decision,
    "clean_baseline_reproduced": clean_reproduced,
    "colour_is_required": colour_is_required,
    "fine_structure_dependency_detected": (
        fine_structure_dependency_detected
    ),
    "global_layout_required_for_recoverability": (
        global_layout_required_for_recoverability
    ),
    "clean_axis_stable_after_patch_shuffle": (
        clean_axis_stable_after_patch_shuffle
    ),
    "primary_fixed_probe_blur_result": {
        "clean_correct": int(
            fixed_clean["correct_pairs"]
        ),
        "blur_correct": int(
            fixed_blur["correct_pairs"]
        ),
        "accuracy_change": float(
            fixed_blur[
                "condition_minus_clean_accuracy"
            ]
        ),
        "accuracy_change_95ci": [
            float(
                fixed_blur[
                    "difference_ci_lower"
                ]
            ),
            float(
                fixed_blur[
                    "difference_ci_upper"
                ]
            ),
        ],
        "exact_mcnemar_p": float(
            fixed_blur["exact_mcnemar_p"]
        ),
    },
    "bounded_conclusion": (
        "Within DeepDRiD, the frozen representation's "
        "eye-local DR severity-direction signal is preserved "
        "after colour removal, disrupted by strong blur, and "
        "recoverable after coarse patch shuffling. The result "
        "is most consistent with fine-scale local achromatic "
        "morphology or texture."
    ),
    "not_yet_resolved": [
        "Whether the signal originates from DR lesions",
        "Whether the signal originates from vascular edges",
        "Whether image sharpness or focus contributes",
        "Which retinal regions carry the signal",
        "Whether the result replicates in another dataset",
    ],
    "next_gate": (
        "Fine-structure dissection with blur dose response, "
        "frequency controls and spatial masking"
    ),
}


# ------------------------------------------------------------
# 5. Save
# ------------------------------------------------------------

MECHANISM_TABLE_PATH = (
    KILL_TEST_ROOT
    / "Locality_Kill_Test_v0.1_Mechanism_Evidence.csv"
)

DECISION_PATH = (
    KILL_TEST_ROOT
    / "Locality_Kill_Test_v0.1_Decision.json"
)

REPORT_PATH = (
    KILL_TEST_ROOT
    / "Locality_Kill_Test_v0.1_Results_and_Decision.md"
)

mechanism_table.to_csv(
    MECHANISM_TABLE_PATH,
    index=False,
)

with open(
    DECISION_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        decision,
        file,
        indent=2,
    )


report = f"""# Retinal DR Locality Kill Test v0.1

**Decision:** `{kill_test_decision}`

## Frozen baseline

- Clean paired accuracy: {int(fixed_clean["correct_pairs"])}/45
- Clean directional AUC: {fixed_clean["validation_directional_auc"]:.4f}

## Colour removal

- Fixed-probe accuracy: {int(fixed_gray["correct_pairs"])}/45
- Directional AUC: {fixed_gray["validation_directional_auc"]:.4f}
- Exact McNemar p versus clean: {fixed_gray["exact_mcnemar_p"]:.6f}

Colour removal did not reduce paired accuracy.

## Strong blur

- Adaptive-probe accuracy: {int(adaptive_blur["correct_pairs"])}/45
- Adaptive directional AUC: {adaptive_blur["validation_directional_auc"]:.4f}
- Fixed-probe accuracy: {int(fixed_blur["correct_pairs"])}/45
- Fixed directional AUC: {fixed_blur["validation_directional_auc"]:.4f}
- Fixed accuracy change versus clean: {fixed_blur["condition_minus_clean_accuracy"]:.4f}
- Paired 95% CI: [{fixed_blur["difference_ci_lower"]:.4f}, {fixed_blur["difference_ci_upper"]:.4f}]
- Exact McNemar p: {fixed_blur["exact_mcnemar_p"]:.6f}

Strong blur significantly disrupted the clean eye-local decision signal,
and the signal was not recovered by condition-specific retraining.

## Patch shuffle

- Adaptive-probe accuracy: {int(adaptive_shuffle["correct_pairs"])}/45
- Adaptive directional AUC: {adaptive_shuffle["validation_directional_auc"]:.4f}
- Fixed-probe accuracy: {int(fixed_shuffle["correct_pairs"])}/45
- Fixed directional AUC: {fixed_shuffle["validation_directional_auc"]:.4f}

The information remained recoverable after global patch shuffling, although
the original clean decision axis did not transfer fully.

## Bounded interpretation

Within DeepDRiD, the eye-local DR severity-direction signal is more
consistent with fine-scale local achromatic morphology or texture than
with colour or intact global retinal topology.

This result does not yet identify a lesion-specific mechanism and does not
exclude vascular edges, focus, sharpness or other high-frequency cues.
Independent dataset replication remains required.
"""

with open(
    REPORT_PATH,
    "w",
    encoding="utf-8",
) as file:
    file.write(report)


# ------------------------------------------------------------
# 6. Final report
# ------------------------------------------------------------

print(
    "================ LOCALITY KILL TEST "
    "FINAL DECISION ================"
)

print("Decision:", kill_test_decision)

print(
    "\nClean baseline reproduced:",
    clean_reproduced,
)

print(
    "Colour required:",
    colour_is_required,
)

print(
    "Fine-structure dependency detected:",
    fine_structure_dependency_detected,
)

print(
    "Global layout required for recoverability:",
    global_layout_required_for_recoverability,
)

print(
    "Clean axis stable after patch shuffle:",
    clean_axis_stable_after_patch_shuffle,
)

print("\nSaved:")
print(MECHANISM_TABLE_PATH)
print(DECISION_PATH)
print(REPORT_PATH)

print(
    "\nLocality Kill Test v0.1 finalized."
)

================ LOCALITY KILL TEST FINAL DECISION ================
Decision: PASS_FINE_SCALE_ACHROMATIC_STRUCTURE_DEPENDENCY

Clean baseline reproduced: True
Colour required: False
Fine-structure dependency detected: True
Global layout required for recoverability: False
Clean axis stable after patch shuffle: False

Saved:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Locality_Kill_Test_v0.1/Locality_Kill_Test_v0.1_Mechanism_Evidence.csv
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Locality_Kill_Test_v0.1/Locality_Kill_Test_v0.1_Decision.json
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Locality_Kill_Test_v0.1/Locality_Kill_Test_v0.1_Results_and_Decision.md

Locality Kill Test v0.1 finalized.


# Retinal DR Frozen Representation Baseline v0.1

**Step 6 of the Retinal DR smoke-test plan**

Purpose: test whether frozen image representations recover eye-local diagnostic information beyond acquisition, quality, resolution and patient-level baselines.

## 00. Runtime, Drive mount and configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(
    '/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability'
)

DEEPDRID_ROOT = (
    PROJECT_ROOT / '02_Dataset_Map' /
    'DeepDRiD_Official_Raw' / 'DeepDRiD_v1.1_Extracted'
)

OUTPUT_ROOT = (
    PROJECT_ROOT / '06_Data_Records' / 'Retinal_DR' /
    'Frozen_Representation_Baseline_v0.1'
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('DeepDRiD root:', DEEPDRID_ROOT)
print('Output root:', OUTPUT_ROOT)

DeepDRiD root: /content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/02_Dataset_Map/DeepDRiD_Official_Raw/DeepDRiD_v1.1_Extracted
Output root: /content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Frozen_Representation_Baseline_v0.1


## 01. Load frozen metadata and define patient-safe evaluation

In [ ]:
from pathlib import Path

METADATA_PATH = (
    PROJECT_ROOT / "06_Data_Records" / "DeepDRiD" /
    "Smoke_Test_v0.1" / "20260718T220050Z" /
    "deepdrid_canonical_metadata.csv"
)

print("DeepDRiD exists:", DEEPDRID_ROOT.exists())
print("Metadata exists:", METADATA_PATH.exists())
print("Metadata path:", METADATA_PATH)

assert DEEPDRID_ROOT.exists(), DEEPDRID_ROOT
assert METADATA_PATH.exists(), METADATA_PATH

DeepDRiD exists: True
Metadata exists: True
Metadata path: /content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/DeepDRiD/Smoke_Test_v0.1/20260718T220050Z/deepdrid_canonical_metadata.csv


In [ ]:
import pandas as pd

meta = pd.read_csv(METADATA_PATH)

print("Shape:", meta.shape)
print("\nColumns:")
for i, col in enumerate(meta.columns):
    print(f"{i:02d}: {col}")

print("\nFirst 3 rows:")
display(meta.head(3))

Shape: (1600, 20)

Columns:
00: patient_id
01: image_id
02: image_path
03: Overall quality
04: left_eye_DR_Level
05: right_eye_DR_Level
06: patient_DR_Level
07: Clarity
08: Field definition
09: Artifact
10: view
11: Source
12: split
13: eye_from_filename
14: view_number
15: eye_resolved
16: eye_name_conflict
17: eye_DR_Level
18: image_path_disk
19: image_exists

First 3 rows:


,patient_id,image_id,image_path,Overall quality,left_eye_DR_Level,right_eye_DR_Level,patient_DR_Level,Clarity,Field definition,Artifact,view,Source,split,eye_from_filename,view_number,eye_resolved,eye_name_conflict,eye_DR_Level,image_path_disk,image_exists
0,1,1_l1,\regular-fundus-training\1\1_l1.jpg,0,0.0,NaN,0,8,8,4,l1,Nicheng,training,l,1,l,False,0.0,/content/drive/MyDrive/Cross-Modal_Diagnostic_...,True
1,1,1_l2,\regular-fundus-training\1\1_l2.jpg,0,0.0,NaN,0,8,8,0,l2,Nicheng,training,l,2,l,False,0.0,/content/drive/MyDrive/Cross-Modal_Diagnostic_...,True
2,1,1_r1,\regular-fundus-training\1\1_r1.jpg,0,NaN,0.0,0,8,8,4,r1,Nicheng,training,r,1,r,False,0.0,/content/drive/MyDrive/Cross-Modal_Diagnostic_...,True


In [ ]:
import numpy as np
import pandas as pd

analysis_df = meta.copy()

# Standardise types
analysis_df["patient_id"] = analysis_df["patient_id"].astype(str)
analysis_df["eye_DR_Level"] = pd.to_numeric(
    analysis_df["eye_DR_Level"], errors="coerce"
)
analysis_df["Overall quality"] = pd.to_numeric(
    analysis_df["Overall quality"], errors="coerce"
)

# Keep only existing, gradable regular-fundus images
analysis_df = analysis_df[
    analysis_df["image_exists"].eq(True)
    & analysis_df["eye_DR_Level"].between(0, 4, inclusive="both")
    & analysis_df["eye_resolved"].isin(["l", "r"])
].copy()

# Binary endpoint used for the first frozen-representation test
analysis_df["referable_DR"] = (
    analysis_df["eye_DR_Level"] >= 2
).astype(int)

train_df = analysis_df[analysis_df["split"] == "training"].copy()
val_df = analysis_df[analysis_df["split"] == "validation"].copy()

train_patients = set(train_df["patient_id"])
val_patients = set(val_df["patient_id"])

print("Analysis images:", len(analysis_df))
print("Analysis patients:", analysis_df["patient_id"].nunique())

print("\nTraining:")
print(" images:", len(train_df))
print(" patients:", train_df["patient_id"].nunique())
print(" referable prevalence:", round(train_df["referable_DR"].mean(), 4))

print("\nValidation:")
print(" images:", len(val_df))
print(" patients:", val_df["patient_id"].nunique())
print(" referable prevalence:", round(val_df["referable_DR"].mean(), 4))

print("\nTrain–validation patient overlap:",
      len(train_patients & val_patients))

print("\nEye-name conflicts retained for audit:",
      int(analysis_df["eye_name_conflict"].sum()))

print("\nDR grade counts by split:")
display(
    pd.crosstab(
        analysis_df["eye_DR_Level"],
        analysis_df["split"]
    )
)

Analysis images: 1600
Analysis patients: 400

Training:
 images: 1200
 patients: 300
 referable prevalence: 0.4333

Validation:
 images: 400
 patients: 100
 referable prevalence: 0.45

Train–validation patient overlap: 0

Eye-name conflicts retained for audit: 11

DR grade counts by split:


split,training,validation
eye_DR_Level,,
0.0,540,174
1.0,140,46
2.0,234,92
3.0,214,68
4.0,72,20


## 02. Frozen encoder and embedding extraction

In [ ]:
import torch
import torchvision
from torch import nn
from torchvision.models import resnet50, ResNet50_Weights

print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Official pretrained weights and matched preprocessing
weights = ResNet50_Weights.DEFAULT
preprocess = weights.transforms()

# Remove the ImageNet classifier and retain the 2048-dimensional representation
encoder = resnet50(weights=weights)
embedding_dim = encoder.fc.in_features
encoder.fc = nn.Identity()

# Strictly frozen inference model
encoder.eval()
for parameter in encoder.parameters():
    parameter.requires_grad = False

encoder = encoder.to(device)

trainable_parameters = sum(
    p.numel() for p in encoder.parameters() if p.requires_grad
)

print("Encoder: ResNet-50 / ImageNet-1K V2")
print("Embedding dimension:", embedding_dim)
print("Trainable parameters:", trainable_parameters)
print("Frozen encoder ready:", trainable_parameters == 0)

PyTorch: 2.11.0+cu128
TorchVision: 0.26.0+cu128
Device: cuda
GPU: Tesla T4
Encoder: ResNet-50 / ImageNet-1K V2
Embedding dimension: 2048
Trainable parameters: 0
Frozen encoder ready: True


In [ ]:
#@title 02.2 Extract and cache frozen ResNet-50 embeddings

from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

import numpy as np
import pandas as pd
import torch


# ------------------------------------------------------------
# 1. Freeze the exact sample order used for embedding extraction
# ------------------------------------------------------------

embedding_meta = analysis_df.reset_index(drop=True).copy()
embedding_meta["embedding_row"] = np.arange(len(embedding_meta))

PATH_COLUMN = "image_path_disk"

required_columns = [
    "patient_id",
    "image_id",
    "split",
    "eye_resolved",
    "eye_DR_Level",
    PATH_COLUMN,
]

missing_columns = [
    column for column in required_columns
    if column not in embedding_meta.columns
]

assert not missing_columns, (
    f"Missing required metadata columns: {missing_columns}"
)

embedding_meta[PATH_COLUMN] = (
    embedding_meta[PATH_COLUMN].astype(str)
)

missing_image_mask = ~embedding_meta[PATH_COLUMN].map(
    lambda path: Path(path).is_file()
)

print("Images scheduled for extraction:", len(embedding_meta))
print("Missing image files:", int(missing_image_mask.sum()))

assert len(embedding_meta) == 1600
assert not missing_image_mask.any()


# ------------------------------------------------------------
# 2. Dataset
# ------------------------------------------------------------

class DeepDRiDEmbeddingDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.image_paths = dataframe[PATH_COLUMN].tolist()
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        image_path = self.image_paths[index]

        try:
            with Image.open(image_path) as image:
                image = image.convert("RGB")
                image_tensor = self.transform(image)

        except Exception as error:
            raise RuntimeError(
                f"Failed to load image at row {index}: "
                f"{image_path}"
            ) from error

        return image_tensor, index


dataset = DeepDRiDEmbeddingDataset(
    dataframe=embedding_meta,
    transform=preprocess,
)


# ------------------------------------------------------------
# 3. DataLoader
# ------------------------------------------------------------

BATCH_SIZE = 32
NUM_WORKERS = 2

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=(NUM_WORKERS > 0),
)

print("Batches:", len(loader))
print("Batch size:", BATCH_SIZE)
print("Workers:", NUM_WORKERS)


# ------------------------------------------------------------
# 4. Output paths
# ------------------------------------------------------------

EMBEDDING_PATH = (
    OUTPUT_ROOT
    / "ResNet50_ImageNet1K_V2_Embeddings_float32.npy"
)

EMBEDDING_INDEX_PATH = (
    OUTPUT_ROOT
    / "ResNet50_ImageNet1K_V2_Embedding_Index.csv"
)

device_type = (
    device.type
    if hasattr(device, "type")
    else str(device).split(":")[0]
)


# ------------------------------------------------------------
# 5. Extract or load cached embeddings
# ------------------------------------------------------------

cache_is_complete = (
    EMBEDDING_PATH.is_file()
    and EMBEDDING_INDEX_PATH.is_file()
)

if cache_is_complete:
    print("\nExisting embedding cache detected.")
    print("Loading:", EMBEDDING_PATH)

    embeddings = np.load(EMBEDDING_PATH)

else:
    print("\nExtracting frozen embeddings...")

    embeddings = np.empty(
        (len(embedding_meta), embedding_dim),
        dtype=np.float32,
    )

    encoder.eval()

    with torch.inference_mode():
        for image_batch, row_indices in tqdm(
            loader,
            desc="Frozen ResNet-50",
        ):
            image_batch = image_batch.to(
                device,
                non_blocking=True,
            )

            with torch.autocast(
                device_type=device_type,
                dtype=torch.float16,
                enabled=(device_type == "cuda"),
            ):
                batch_embeddings = encoder(image_batch)

            batch_embeddings = (
                batch_embeddings
                .float()
                .cpu()
                .numpy()
            )

            row_indices = row_indices.numpy()

            embeddings[row_indices] = batch_embeddings

    np.save(
        EMBEDDING_PATH,
        embeddings,
        allow_pickle=False,
    )

    embedding_meta.to_csv(
        EMBEDDING_INDEX_PATH,
        index=False,
    )

    print("\nEmbedding extraction completed.")
    print("Saved embeddings:", EMBEDDING_PATH)
    print("Saved index:", EMBEDDING_INDEX_PATH)


# ------------------------------------------------------------
# 6. Embedding integrity checks
# ------------------------------------------------------------

assert embeddings.shape == (
    len(embedding_meta),
    embedding_dim,
), embeddings.shape

assert embeddings.dtype == np.float32
assert np.isfinite(embeddings).all()

embedding_norms = np.linalg.norm(
    embeddings,
    axis=1,
)

dimension_std = embeddings.std(axis=0)

print("\n================ EMBEDDING AUDIT ================")
print("Embedding shape:", embeddings.shape)
print("Embedding dtype:", embeddings.dtype)
print("Finite values:", bool(np.isfinite(embeddings).all()))

print(
    "Embedding norm — "
    f"min: {embedding_norms.min():.4f}, "
    f"mean: {embedding_norms.mean():.4f}, "
    f"max: {embedding_norms.max():.4f}"
)

print(
    "Dimension SD — "
    f"min: {dimension_std.min():.6f}, "
    f"median: {np.median(dimension_std):.6f}, "
    f"max: {dimension_std.max():.6f}"
)

print(
    "Near-zero-variance dimensions:",
    int((dimension_std < 1e-8).sum()),
)

print("\nRows by split:")
print(embedding_meta.groupby("split").size())

assert embedding_norms.min() > 0
assert (dimension_std > 1e-8).any()

print("\nFrozen embedding extraction passed.")

Images scheduled for extraction: 1600
Missing image files: 0
Batches: 50
Batch size: 32
Workers: 2

Extracting frozen embeddings...


Frozen ResNet-50:   0%|          | 0/50 [00:00<?, ?it/s]


Embedding extraction completed.
Saved embeddings: /content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Frozen_Representation_Baseline_v0.1/ResNet50_ImageNet1K_V2_Embeddings_float32.npy
Saved index: /content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Frozen_Representation_Baseline_v0.1/ResNet50_ImageNet1K_V2_Embedding_Index.csv

================ EMBEDDING AUDIT ================
Embedding shape: (1600, 2048)
Embedding dtype: float32
Finite values: True
Embedding norm — min: 8.5410, mean: 11.7883, max: 16.0624
Dimension SD — min: 0.000000, median: 0.029314, max: 1.043323
Near-zero-variance dimensions: 1

Rows by split:
split
training      1200
validation     400
dtype: int64

Frozen embedding extraction passed.


## 03. Patient-grouped diagnostic baseline

In [ ]:
#@title 03.1 Patient-grouped eye-level frozen linear probe

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.base import clone

import json
import joblib
import numpy as np
import pandas as pd


RANDOM_SEED = 20260719


# ------------------------------------------------------------
# 1. Aggregate the two views of each eye
# ------------------------------------------------------------

group_columns = [
    "split",
    "patient_id",
    "eye_resolved",
]

eye_records = []
eye_embedding_list = []

grouped_rows = embedding_meta.groupby(
    group_columns,
    sort=False,
).groups

for group_key, row_indices in grouped_rows.items():
    split_name, patient_id, eye_name = group_key

    row_indices = np.asarray(
        list(row_indices),
        dtype=int,
    )

    group_metadata = embedding_meta.iloc[row_indices]

    eye_grades = (
        group_metadata["eye_DR_Level"]
        .dropna()
        .astype(int)
        .unique()
    )

    if len(eye_grades) != 1:
        raise ValueError(
            f"Inconsistent eye grades for "
            f"{split_name}, patient={patient_id}, eye={eye_name}: "
            f"{eye_grades}"
        )

    source_values = (
        group_metadata["Source"]
        .dropna()
        .astype(str)
        .unique()
    )

    if len(source_values) != 1:
        raise ValueError(
            f"Inconsistent acquisition source for "
            f"{split_name}, patient={patient_id}, eye={eye_name}: "
            f"{source_values}"
        )

    eye_grade = int(eye_grades[0])

    eye_records.append(
        {
            "split": split_name,
            "patient_id": str(patient_id),
            "eye_resolved": eye_name,
            "eye_DR_Level": eye_grade,
            "referable_dr": int(eye_grade >= 2),
            "Source": source_values[0],
            "number_of_views": len(row_indices),
            "image_ids": "|".join(
                group_metadata["image_id"].astype(str)
            ),
            "mean_overall_quality": float(
                group_metadata["Overall quality"].mean()
            ),
            "mean_clarity": float(
                group_metadata["Clarity"].mean()
            ),
            "mean_field_definition": float(
                group_metadata["Field definition"].mean()
            ),
            "mean_artifact": float(
                group_metadata["Artifact"].mean()
            ),
        }
    )

    eye_embedding_list.append(
        embeddings[row_indices].mean(axis=0)
    )


eye_meta = pd.DataFrame(eye_records)

eye_embeddings = np.vstack(
    eye_embedding_list
).astype(np.float32)


# ------------------------------------------------------------
# 2. Eye-level integrity checks
# ------------------------------------------------------------

print("================ EYE-LEVEL DATA AUDIT ================")

print("Eye embeddings:", eye_embeddings.shape)

print("\nEyes by split:")
print(eye_meta.groupby("split").size())

print("\nPatients by split:")
print(
    eye_meta.groupby("split")["patient_id"]
    .nunique()
)

print("\nViews per eye:")
print(
    eye_meta["number_of_views"]
    .value_counts()
    .sort_index()
)

print("\nReferable-DR prevalence:")
print(
    eye_meta.groupby("split")["referable_dr"]
    .agg(["count", "mean"])
)

assert eye_embeddings.shape == (800, embedding_dim)
assert len(eye_meta) == 800

assert (
    eye_meta.groupby(["split", "patient_id"])
    .size()
    .eq(2)
    .all()
), "Every patient should contribute two eyes."

assert (
    eye_meta["number_of_views"] == 2
).all(), "Every eye should contain two views."

assert (
    eye_meta.loc[
        eye_meta["split"].eq("training"),
        "patient_id",
    ].nunique()
    == 300
)

assert (
    eye_meta.loc[
        eye_meta["split"].eq("validation"),
        "patient_id",
    ].nunique()
    == 100
)


# ------------------------------------------------------------
# 3. Define official train and validation partitions
# ------------------------------------------------------------

train_mask = eye_meta["split"].eq("training").to_numpy()
validation_mask = eye_meta["split"].eq("validation").to_numpy()

X_train = eye_embeddings[train_mask]
X_validation = eye_embeddings[validation_mask]

y_train = (
    eye_meta.loc[train_mask, "referable_dr"]
    .to_numpy(dtype=int)
)

y_validation = (
    eye_meta.loc[validation_mask, "referable_dr"]
    .to_numpy(dtype=int)
)

training_groups = (
    eye_meta.loc[train_mask, "patient_id"]
    .astype(str)
    .to_numpy()
)

validation_meta = (
    eye_meta.loc[validation_mask]
    .reset_index(drop=True)
    .copy()
)


# ------------------------------------------------------------
# 4. Patient-grouped hyperparameter selection
# ------------------------------------------------------------

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_SEED,
)

C_GRID = [
    1e-4,
    3e-4,
    1e-3,
    3e-3,
    1e-2,
    3e-2,
    1e-1,
    3e-1,
    1.0,
]

cv_rows = []

for C_value in C_GRID:
    fold_aucs = []

    for fold_number, (
        fold_train_indices,
        fold_test_indices,
    ) in enumerate(
        cv.split(
            X_train,
            y_train,
            groups=training_groups,
        ),
        start=1,
    ):
        model = Pipeline(
            steps=[
                (
                    "standardize",
                    StandardScaler(),
                ),
                (
                    "classifier",
                    LogisticRegression(
                        penalty="l2",
                        C=C_value,
                        solver="liblinear",
                        max_iter=10000,
                        random_state=RANDOM_SEED,
                    ),
                ),
            ]
        )

        model.fit(
            X_train[fold_train_indices],
            y_train[fold_train_indices],
        )

        fold_probabilities = model.predict_proba(
            X_train[fold_test_indices]
        )[:, 1]

        fold_auc = roc_auc_score(
            y_train[fold_test_indices],
            fold_probabilities,
        )

        fold_aucs.append(float(fold_auc))

    cv_rows.append(
        {
            "C": C_value,
            "mean_grouped_cv_auc": np.mean(fold_aucs),
            "sd_grouped_cv_auc": np.std(
                fold_aucs,
                ddof=1,
            ),
            **{
                f"fold_{index + 1}_auc": value
                for index, value in enumerate(fold_aucs)
            },
        }
    )


cv_results = pd.DataFrame(cv_rows)

cv_results = cv_results.sort_values(
    by=[
        "mean_grouped_cv_auc",
        "C",
    ],
    ascending=[
        False,
        True,
    ],
).reset_index(drop=True)

best_C = float(cv_results.loc[0, "C"])

print("\n================ GROUPED CV RESULTS ================")

display(
    cv_results[
        [
            "C",
            "mean_grouped_cv_auc",
            "sd_grouped_cv_auc",
        ]
    ]
)

print("Selected C:", best_C)


# ------------------------------------------------------------
# 5. Generate patient-grouped out-of-fold predictions
# ------------------------------------------------------------

best_probe_template = Pipeline(
    steps=[
        (
            "standardize",
            StandardScaler(),
        ),
        (
            "classifier",
            LogisticRegression(
                penalty="l2",
                C=best_C,
                solver="liblinear",
                max_iter=10000,
                random_state=RANDOM_SEED,
            ),
        ),
    ]
)

training_oof_probability = np.full(
    len(y_train),
    np.nan,
    dtype=float,
)

for fold_train_indices, fold_test_indices in cv.split(
    X_train,
    y_train,
    groups=training_groups,
):
    fold_model = clone(best_probe_template)

    fold_model.fit(
        X_train[fold_train_indices],
        y_train[fold_train_indices],
    )

    training_oof_probability[fold_test_indices] = (
        fold_model.predict_proba(
            X_train[fold_test_indices]
        )[:, 1]
    )

assert np.isfinite(training_oof_probability).all()

training_oof_auc = roc_auc_score(
    y_train,
    training_oof_probability,
)

training_oof_ap = average_precision_score(
    y_train,
    training_oof_probability,
)


# ------------------------------------------------------------
# 6. Fit on all official training patients
# ------------------------------------------------------------

final_probe = clone(best_probe_template)

final_probe.fit(
    X_train,
    y_train,
)

validation_probability = final_probe.predict_proba(
    X_validation
)[:, 1]

validation_auc = roc_auc_score(
    y_validation,
    validation_probability,
)

validation_ap = average_precision_score(
    y_validation,
    validation_probability,
)


# ------------------------------------------------------------
# 7. Patient-clustered bootstrap confidence interval
# ------------------------------------------------------------

def clustered_patient_bootstrap_auc(
    metadata,
    labels,
    probabilities,
    number_of_bootstraps=2000,
    random_seed=RANDOM_SEED,
):
    rng = np.random.default_rng(random_seed)

    patient_ids = (
        metadata["patient_id"]
        .astype(str)
        .unique()
    )

    metadata_patient_ids = (
        metadata["patient_id"]
        .astype(str)
        .to_numpy()
    )

    patient_to_indices = {
        patient_id: np.flatnonzero(
            metadata_patient_ids == patient_id
        )
        for patient_id in patient_ids
    }

    bootstrap_aucs = []

    for _ in range(number_of_bootstraps):
        sampled_patients = rng.choice(
            patient_ids,
            size=len(patient_ids),
            replace=True,
        )

        sampled_indices = np.concatenate(
            [
                patient_to_indices[patient_id]
                for patient_id in sampled_patients
            ]
        )

        sampled_labels = labels[sampled_indices]
        sampled_probabilities = probabilities[sampled_indices]

        if np.unique(sampled_labels).size < 2:
            continue

        bootstrap_aucs.append(
            roc_auc_score(
                sampled_labels,
                sampled_probabilities,
            )
        )

    bootstrap_aucs = np.asarray(
        bootstrap_aucs,
        dtype=float,
    )

    return {
        "lower": float(
            np.percentile(bootstrap_aucs, 2.5)
        ),
        "upper": float(
            np.percentile(bootstrap_aucs, 97.5)
        ),
        "valid_bootstraps": int(
            len(bootstrap_aucs)
        ),
    }


validation_auc_ci = clustered_patient_bootstrap_auc(
    metadata=validation_meta,
    labels=y_validation,
    probabilities=validation_probability,
)


# ------------------------------------------------------------
# 8. Save predictions and model artifacts
# ------------------------------------------------------------

eye_meta["probe_probability"] = np.nan
eye_meta["prediction_type"] = ""

eye_meta.loc[
    train_mask,
    "probe_probability",
] = training_oof_probability

eye_meta.loc[
    train_mask,
    "prediction_type",
] = "patient-grouped out-of-fold"

eye_meta.loc[
    validation_mask,
    "probe_probability",
] = validation_probability

eye_meta.loc[
    validation_mask,
    "prediction_type",
] = "official held-out validation"


EYE_EMBEDDING_PATH = (
    OUTPUT_ROOT
    / "ResNet50_ImageNet1K_V2_EyeMean_Embeddings_float32.npy"
)

EYE_INDEX_PATH = (
    OUTPUT_ROOT
    / "ResNet50_ImageNet1K_V2_EyeMean_Index_and_Predictions.csv"
)

CV_RESULTS_PATH = (
    OUTPUT_ROOT
    / "PatientGrouped_LinearProbe_CV_Results.csv"
)

MODEL_PATH = (
    OUTPUT_ROOT
    / "PatientGrouped_LinearProbe.joblib"
)

SUMMARY_PATH = (
    OUTPUT_ROOT
    / "PatientGrouped_LinearProbe_Summary.json"
)

np.save(
    EYE_EMBEDDING_PATH,
    eye_embeddings,
    allow_pickle=False,
)

eye_meta.to_csv(
    EYE_INDEX_PATH,
    index=False,
)

cv_results.to_csv(
    CV_RESULTS_PATH,
    index=False,
)

joblib.dump(
    final_probe,
    MODEL_PATH,
)

summary = {
    "representation": "ResNet-50 ImageNet-1K V2",
    "representation_status": "frozen",
    "analysis_unit": "eye-level mean of two views",
    "training_patients": 300,
    "training_eyes": 600,
    "validation_patients": 100,
    "validation_eyes": 200,
    "selected_C": best_C,
    "training_grouped_oof_auc": float(
        training_oof_auc
    ),
    "training_grouped_oof_average_precision": float(
        training_oof_ap
    ),
    "validation_auc": float(
        validation_auc
    ),
    "validation_average_precision": float(
        validation_ap
    ),
    "validation_auc_clustered_95ci": [
        validation_auc_ci["lower"],
        validation_auc_ci["upper"],
    ],
    "validation_auc_valid_bootstraps": (
        validation_auc_ci["valid_bootstraps"]
    ),
}

with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        summary,
        file,
        indent=2,
    )


# ------------------------------------------------------------
# 9. Final report
# ------------------------------------------------------------

print("\n================ LINEAR PROBE RESULT ================")

print(
    "Training patient-grouped OOF AUC: "
    f"{training_oof_auc:.4f}"
)

print(
    "Training patient-grouped OOF AP: "
    f"{training_oof_ap:.4f}"
)

print(
    "\nOfficial validation AUC: "
    f"{validation_auc:.4f}"
)

print(
    "Official validation AP: "
    f"{validation_ap:.4f}"
)

print(
    "Patient-clustered 95% AUC CI: "
    f"[{validation_auc_ci['lower']:.4f}, "
    f"{validation_auc_ci['upper']:.4f}]"
)

print("\nSaved:")
print(EYE_EMBEDDING_PATH)
print(EYE_INDEX_PATH)
print(CV_RESULTS_PATH)
print(MODEL_PATH)
print(SUMMARY_PATH)

print("\nPatient-grouped frozen linear probe passed.")

================ EYE-LEVEL DATA AUDIT ================
Eye embeddings: (800, 2048)

Eyes by split:
split
training      600
validation    200
dtype: int64

Patients by split:
split
training      300
validation    100
Name: patient_id, dtype: int64

Views per eye:
number_of_views
2    800
Name: count, dtype: int64

Referable-DR prevalence:
            count      mean
split                      
training      600  0.433333
validation    200  0.450000

================ GROUPED CV RESULTS ================


,C,mean_grouped_cv_auc,sd_grouped_cv_auc
0,0.0030,0.915825,0.024845
1,0.0100,0.913819,0.022169
2,0.0010,0.913649,0.025701
3,0.0300,0.912246,0.021106
4,0.1000,0.909417,0.021196
5,0.3000,0.907724,0.019445
6,1.0000,0.906549,0.019499
7,0.0003,0.904437,0.031248
8,0.0001,0.889034,0.036364


Selected C: 0.003

================ LINEAR PROBE RESULT ================
Training patient-grouped OOF AUC: 0.9143
Training patient-grouped OOF AP: 0.8896

Official validation AUC: 0.9392
Official validation AP: 0.9428
Patient-clustered 95% AUC CI: [0.8968, 0.9738]

Saved:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Frozen_Representation_Baseline_v0.1/ResNet50_ImageNet1K_V2_EyeMean_Embeddings_float32.npy
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Frozen_Representation_Baseline_v0.1/ResNet50_ImageNet1K_V2_EyeMean_Index_and_Predictions.csv
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Frozen_Representation_Baseline_v0.1/PatientGrouped_LinearProbe_CV_Results.csv
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Frozen_Representation_Baseline_v0.1/PatientGrouped_LinearProbe.joblib
/content/drive/MyDrive/Cross-Modal_Diagnostic_Ob

## 04. Source- and resolution-stratified evaluation

In [ ]:
#@title 04.1 Audit held-out performance within source and resolution strata

from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score


# ------------------------------------------------------------
# 1. Read image dimensions without decoding full images
# ------------------------------------------------------------

IMAGE_SIZE_CACHE_PATH = (
    OUTPUT_ROOT
    / "DeepDRiD_Image_Size_Audit.csv"
)

if IMAGE_SIZE_CACHE_PATH.is_file():
    print("Loading cached image-size audit:")
    print(IMAGE_SIZE_CACHE_PATH)

    image_size_df = pd.read_csv(
        IMAGE_SIZE_CACHE_PATH
    )

else:
    print("Reading image dimensions...")

    size_records = []

    for row in tqdm(
        embedding_meta.itertuples(index=False),
        total=len(embedding_meta),
        desc="Image dimensions",
    ):
        image_path = Path(
            getattr(row, PATH_COLUMN)
        )

        try:
            with Image.open(image_path) as image:
                width, height = image.size

        except Exception as error:
            raise RuntimeError(
                f"Could not read image dimensions: "
                f"{image_path}"
            ) from error

        size_records.append(
            {
                "embedding_row": int(
                    row.embedding_row
                ),
                "width": int(width),
                "height": int(height),
                "resolution": (
                    f"{int(width)}x{int(height)}"
                ),
            }
        )

    image_size_df = pd.DataFrame(
        size_records
    )

    image_size_df.to_csv(
        IMAGE_SIZE_CACHE_PATH,
        index=False,
    )

    print("Saved image-size audit:")
    print(IMAGE_SIZE_CACHE_PATH)


assert len(image_size_df) == len(embedding_meta)
assert image_size_df["embedding_row"].is_unique


# ------------------------------------------------------------
# 2. Attach dimensions to image-level metadata
# ------------------------------------------------------------

image_metadata_with_size = embedding_meta.merge(
    image_size_df,
    on="embedding_row",
    how="left",
    validate="one_to_one",
)

assert image_metadata_with_size[
    "resolution"
].notna().all()

image_metadata_with_size["patient_id"] = (
    image_metadata_with_size["patient_id"]
    .astype(str)
)


# ------------------------------------------------------------
# 3. Resolve one acquisition resolution per eye
# ------------------------------------------------------------

eye_size_records = []

for (
    split_name,
    patient_id,
    eye_name,
), group in image_metadata_with_size.groupby(
    [
        "split",
        "patient_id",
        "eye_resolved",
    ],
    sort=False,
):
    resolutions = sorted(
        group["resolution"]
        .dropna()
        .astype(str)
        .unique()
    )

    widths = sorted(
        group["width"]
        .dropna()
        .astype(int)
        .unique()
    )

    heights = sorted(
        group["height"]
        .dropna()
        .astype(int)
        .unique()
    )

    resolution_consistent = (
        len(resolutions) == 1
    )

    eye_size_records.append(
        {
            "split": split_name,
            "patient_id": str(patient_id),
            "eye_resolved": eye_name,
            "resolution": (
                resolutions[0]
                if resolution_consistent
                else "MIXED:" + "|".join(resolutions)
            ),
            "width": (
                widths[0]
                if len(widths) == 1
                else np.nan
            ),
            "height": (
                heights[0]
                if len(heights) == 1
                else np.nan
            ),
            "resolution_consistent_within_eye": (
                resolution_consistent
            ),
        }
    )


eye_size_df = pd.DataFrame(
    eye_size_records
)

assert len(eye_size_df) == len(eye_meta)


# ------------------------------------------------------------
# 4. Join acquisition strata to eye-level predictions
# ------------------------------------------------------------

eye_meta_stratified = eye_meta.copy()

eye_meta_stratified["patient_id"] = (
    eye_meta_stratified["patient_id"]
    .astype(str)
)

eye_meta_stratified = eye_meta_stratified.merge(
    eye_size_df,
    on=[
        "split",
        "patient_id",
        "eye_resolved",
    ],
    how="left",
    validate="one_to_one",
)

assert eye_meta_stratified[
    "resolution"
].notna().all()

assert eye_meta_stratified[
    "probe_probability"
].notna().all()


print(
    "Eyes with inconsistent resolution "
    "between their two views:",
    int(
        (
            ~eye_meta_stratified[
                "resolution_consistent_within_eye"
            ]
        ).sum()
    ),
)


# ------------------------------------------------------------
# 5. Validation acquisition structure
# ------------------------------------------------------------

validation_stratified = (
    eye_meta_stratified.loc[
        eye_meta_stratified[
            "split"
        ].eq("validation")
    ]
    .reset_index(drop=True)
    .copy()
)


print(
    "\n================ VALIDATION ACQUISITION AUDIT "
    "================"
)

print("\nEyes and patients by source:")

source_counts = (
    validation_stratified
    .groupby("Source")
    .agg(
        eyes=("referable_dr", "size"),
        patients=("patient_id", "nunique"),
        prevalence=("referable_dr", "mean"),
    )
    .sort_values(
        "eyes",
        ascending=False,
    )
)

display(source_counts)


print("\nEyes and patients by resolution:")

resolution_counts = (
    validation_stratified
    .groupby("resolution")
    .agg(
        eyes=("referable_dr", "size"),
        patients=("patient_id", "nunique"),
        prevalence=("referable_dr", "mean"),
    )
    .sort_values(
        "eyes",
        ascending=False,
    )
)

display(resolution_counts)


print("\nSource × resolution table:")

display(
    pd.crosstab(
        validation_stratified["Source"],
        validation_stratified["resolution"],
        margins=True,
    )
)


# ------------------------------------------------------------
# 6. Compute clustered subgroup metrics
# ------------------------------------------------------------

def calculate_stratified_metrics(
    dataframe,
    stratum_column,
    number_of_bootstraps=1000,
):
    rows = []

    for stratum_value, subset in dataframe.groupby(
        stratum_column,
        dropna=False,
    ):
        subset = subset.reset_index(
            drop=True
        )

        labels = subset[
            "referable_dr"
        ].to_numpy(dtype=int)

        probabilities = subset[
            "probe_probability"
        ].to_numpy(dtype=float)

        number_of_classes = (
            np.unique(labels).size
        )

        result = {
            "stratum_type": stratum_column,
            "stratum": str(stratum_value),
            "eyes": int(len(subset)),
            "patients": int(
                subset["patient_id"].nunique()
            ),
            "positive_eyes": int(labels.sum()),
            "negative_eyes": int(
                len(labels) - labels.sum()
            ),
            "prevalence": float(labels.mean()),
            "auc": np.nan,
            "average_precision": np.nan,
            "auc_ci_lower": np.nan,
            "auc_ci_upper": np.nan,
            "valid_bootstraps": 0,
        }

        if number_of_classes == 2:
            result["auc"] = float(
                roc_auc_score(
                    labels,
                    probabilities,
                )
            )

            result["average_precision"] = float(
                average_precision_score(
                    labels,
                    probabilities,
                )
            )

            confidence_interval = (
                clustered_patient_bootstrap_auc(
                    metadata=subset,
                    labels=labels,
                    probabilities=probabilities,
                    number_of_bootstraps=(
                        number_of_bootstraps
                    ),
                    random_seed=RANDOM_SEED,
                )
            )

            result["auc_ci_lower"] = (
                confidence_interval["lower"]
            )

            result["auc_ci_upper"] = (
                confidence_interval["upper"]
            )

            result["valid_bootstraps"] = (
                confidence_interval[
                    "valid_bootstraps"
                ]
            )

        rows.append(result)

    return pd.DataFrame(rows)


source_metrics = calculate_stratified_metrics(
    dataframe=validation_stratified,
    stratum_column="Source",
)

resolution_metrics = calculate_stratified_metrics(
    dataframe=validation_stratified,
    stratum_column="resolution",
)

stratified_metrics = pd.concat(
    [
        source_metrics,
        resolution_metrics,
    ],
    ignore_index=True,
)


# ------------------------------------------------------------
# 7. Save the complete acquisition audit
# ------------------------------------------------------------

STRATIFIED_PREDICTIONS_PATH = (
    OUTPUT_ROOT
    / "Validation_Predictions_with_Source_and_Resolution.csv"
)

STRATIFIED_METRICS_PATH = (
    OUTPUT_ROOT
    / "Validation_Source_and_Resolution_Stratified_Metrics.csv"
)

eye_meta_stratified.to_csv(
    STRATIFIED_PREDICTIONS_PATH,
    index=False,
)

stratified_metrics.to_csv(
    STRATIFIED_METRICS_PATH,
    index=False,
)


# ------------------------------------------------------------
# 8. Final report
# ------------------------------------------------------------

print(
    "\n================ STRATIFIED PERFORMANCE "
    "================"
)

display(
    stratified_metrics[
        [
            "stratum_type",
            "stratum",
            "eyes",
            "patients",
            "prevalence",
            "auc",
            "average_precision",
            "auc_ci_lower",
            "auc_ci_upper",
        ]
    ].sort_values(
        [
            "stratum_type",
            "eyes",
        ],
        ascending=[
            True,
            False,
        ],
    )
)

print("\nSaved:")
print(IMAGE_SIZE_CACHE_PATH)
print(STRATIFIED_PREDICTIONS_PATH)
print(STRATIFIED_METRICS_PATH)

print(
    "\nSource- and resolution-stratified "
    "evaluation passed."
)

Reading image dimensions...


Image dimensions:   0%|          | 0/1600 [00:00<?, ?it/s]

Saved image-size audit:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Frozen_Representation_Baseline_v0.1/DeepDRiD_Image_Size_Audit.csv
Eyes with inconsistent resolution between their two views: 26

================ VALIDATION ACQUISITION AUDIT ================

Eyes and patients by source:


,eyes,patients,prevalence
Source,,,
Nicheng,184,92,0.445652
Nation,10,5,0.400000
Shanghai,6,3,0.666667



Eyes and patients by resolution:


...


Source × resolution table:


resolution,1736x1824,1976x1984,MIXED:1734x1821|1736x1824,All
Source,,,,
Nation,3,6,1,10
Nicheng,135,46,3,184
Shanghai,4,2,0,6
All,142,54,4,200



================ STRATIFIED PERFORMANCE ================


,stratum_type,stratum,eyes,patients,prevalence,auc,average_precision,auc_ci_lower,auc_ci_upper
1,Source,Nicheng,184,92,0.445652,0.934601,0.937718,0.884967,0.973570
0,Source,Nation,10,5,0.400000,1.000000,1.000000,1.000000,1.000000
2,Source,Shanghai,6,3,0.666667,1.000000,1.000000,1.000000,1.000000
3,resolution,1736x1824,142,73,0.338028,0.920656,0.897069,0.850804,0.974640
4,resolution,1976x1984,54,27,0.777778,0.920635,0.978936,0.832640,0.980288
5,resolution,MIXED:1734x1821|1736x1824,4,4,0.000000,NaN,NaN,NaN,NaN



Saved:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Frozen_Representation_Baseline_v0.1/DeepDRiD_Image_Size_Audit.csv
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Frozen_Representation_Baseline_v0.1/Validation_Predictions_with_Source_and_Resolution.csv
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Frozen_Representation_Baseline_v0.1/Validation_Source_and_Resolution_Stratified_Metrics.csv

Source- and resolution-stratified evaluation passed.


## 05. Between-patient versus within-patient decomposition

In [ ]:
#@title 05.1 Between-patient versus within-patient decomposition

import json
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score


# ------------------------------------------------------------
# 1. Prepare the held-out validation predictions
# ------------------------------------------------------------

decomposition_df = (
    validation_stratified
    .reset_index(drop=True)
    .copy()
)

decomposition_df["patient_id"] = (
    decomposition_df["patient_id"]
    .astype(str)
)

assert len(decomposition_df) == 200
assert decomposition_df["patient_id"].nunique() == 100

eyes_per_patient = (
    decomposition_df
    .groupby("patient_id")
    .size()
)

assert eyes_per_patient.eq(2).all(), (
    "Every validation patient must contribute exactly two eyes."
)


# ------------------------------------------------------------
# 2. Decompose each score into patient mean and eye deviation
# ------------------------------------------------------------

decomposition_df["patient_mean_probability"] = (
    decomposition_df
    .groupby("patient_id")["probe_probability"]
    .transform("mean")
)

decomposition_df["within_patient_probability"] = (
    decomposition_df["probe_probability"]
    - decomposition_df["patient_mean_probability"]
)

decomposition_df["patient_referable_count"] = (
    decomposition_df
    .groupby("patient_id")["referable_dr"]
    .transform("sum")
    .astype(int)
)

decomposition_df["patient_mean_grade"] = (
    decomposition_df
    .groupby("patient_id")["eye_DR_Level"]
    .transform("mean")
)


# ------------------------------------------------------------
# 3. Full and between-patient component AUC
# ------------------------------------------------------------

validation_labels = (
    decomposition_df["referable_dr"]
    .to_numpy(dtype=int)
)

full_probabilities = (
    decomposition_df["probe_probability"]
    .to_numpy(dtype=float)
)

between_probabilities = (
    decomposition_df["patient_mean_probability"]
    .to_numpy(dtype=float)
)

full_eye_auc = roc_auc_score(
    validation_labels,
    full_probabilities,
)

between_patient_component_auc = roc_auc_score(
    validation_labels,
    between_probabilities,
)

full_eye_auc_ci = clustered_patient_bootstrap_auc(
    metadata=decomposition_df,
    labels=validation_labels,
    probabilities=full_probabilities,
    number_of_bootstraps=2000,
    random_seed=RANDOM_SEED,
)

between_patient_auc_ci = clustered_patient_bootstrap_auc(
    metadata=decomposition_df,
    labels=validation_labels,
    probabilities=between_probabilities,
    number_of_bootstraps=2000,
    random_seed=RANDOM_SEED + 1,
)


# ------------------------------------------------------------
# 4. Binary within-patient evaluation
#
# Only patients with one referable and one non-referable eye
# contain binary within-patient information.
# ------------------------------------------------------------

binary_discordant_mask = (
    decomposition_df["patient_referable_count"] == 1
)

binary_discordant_df = (
    decomposition_df.loc[binary_discordant_mask]
    .reset_index(drop=True)
    .copy()
)

binary_discordant_patients = (
    binary_discordant_df["patient_id"]
    .nunique()
)

if binary_discordant_patients > 0:
    binary_labels = (
        binary_discordant_df["referable_dr"]
        .to_numpy(dtype=int)
    )

    binary_within_probabilities = (
        binary_discordant_df["within_patient_probability"]
        .to_numpy(dtype=float)
    )

    binary_within_centered_auc = roc_auc_score(
        binary_labels,
        binary_within_probabilities,
    )

    binary_within_auc_ci = clustered_patient_bootstrap_auc(
        metadata=binary_discordant_df,
        labels=binary_labels,
        probabilities=binary_within_probabilities,
        number_of_bootstraps=2000,
        random_seed=RANDOM_SEED + 2,
    )

else:
    binary_within_centered_auc = np.nan

    binary_within_auc_ci = {
        "lower": np.nan,
        "upper": np.nan,
        "valid_bootstraps": 0,
    }


# ------------------------------------------------------------
# 5. Build one matched comparison record per patient
# ------------------------------------------------------------

pair_records = []

for patient_id, patient_group in decomposition_df.groupby(
    "patient_id",
    sort=False,
):
    patient_group = patient_group.reset_index(drop=True)

    assert len(patient_group) == 2

    scores = patient_group[
        "probe_probability"
    ].to_numpy(dtype=float)

    labels = patient_group[
        "referable_dr"
    ].to_numpy(dtype=int)

    grades = patient_group[
        "eye_DR_Level"
    ].to_numpy(dtype=int)

    eyes = patient_group[
        "eye_resolved"
    ].astype(str).to_numpy()

    record = {
        "patient_id": str(patient_id),
        "eye_1": eyes[0],
        "eye_2": eyes[1],
        "eye_1_grade": int(grades[0]),
        "eye_2_grade": int(grades[1]),
        "eye_1_referable": int(labels[0]),
        "eye_2_referable": int(labels[1]),
        "eye_1_probability": float(scores[0]),
        "eye_2_probability": float(scores[1]),
        "binary_discordant": bool(labels[0] != labels[1]),
        "ordinal_discordant": bool(grades[0] != grades[1]),
        "binary_margin": np.nan,
        "binary_pair_correct": np.nan,
        "ordinal_margin": np.nan,
        "ordinal_pair_correct": np.nan,
    }

    # Binary referable-DR matched comparison
    if labels[0] != labels[1]:
        positive_index = int(np.argmax(labels))
        negative_index = 1 - positive_index

        binary_margin = (
            scores[positive_index]
            - scores[negative_index]
        )

        record["binary_margin"] = float(binary_margin)

        record["binary_pair_correct"] = (
            1.0 if binary_margin > 0
            else 0.0 if binary_margin < 0
            else 0.5
        )

    # Ordinal DR-grade matched comparison
    if grades[0] != grades[1]:
        higher_grade_index = int(np.argmax(grades))
        lower_grade_index = 1 - higher_grade_index

        ordinal_margin = (
            scores[higher_grade_index]
            - scores[lower_grade_index]
        )

        record["ordinal_margin"] = float(ordinal_margin)

        record["ordinal_pair_correct"] = (
            1.0 if ordinal_margin > 0
            else 0.0 if ordinal_margin < 0
            else 0.5
        )

    pair_records.append(record)


pairwise_df = pd.DataFrame(pair_records)

binary_pairwise_df = (
    pairwise_df.loc[
        pairwise_df["binary_discordant"]
    ]
    .reset_index(drop=True)
    .copy()
)

ordinal_pairwise_df = (
    pairwise_df.loc[
        pairwise_df["ordinal_discordant"]
    ]
    .reset_index(drop=True)
    .copy()
)


# ------------------------------------------------------------
# 6. Bootstrap matched-pair accuracy by patient
# ------------------------------------------------------------

def bootstrap_pairwise_mean(
    values,
    number_of_bootstraps=5000,
    random_seed=RANDOM_SEED,
):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return {
            "estimate": np.nan,
            "lower": np.nan,
            "upper": np.nan,
            "number_of_pairs": 0,
        }

    rng = np.random.default_rng(random_seed)

    bootstrap_estimates = np.empty(
        number_of_bootstraps,
        dtype=float,
    )

    for bootstrap_index in range(number_of_bootstraps):
        sampled_values = rng.choice(
            values,
            size=len(values),
            replace=True,
        )

        bootstrap_estimates[bootstrap_index] = (
            sampled_values.mean()
        )

    return {
        "estimate": float(values.mean()),
        "lower": float(
            np.percentile(
                bootstrap_estimates,
                2.5,
            )
        ),
        "upper": float(
            np.percentile(
                bootstrap_estimates,
                97.5,
            )
        ),
        "number_of_pairs": int(len(values)),
    }


binary_pairwise_result = bootstrap_pairwise_mean(
    binary_pairwise_df["binary_pair_correct"],
    number_of_bootstraps=5000,
    random_seed=RANDOM_SEED + 3,
)

ordinal_pairwise_result = bootstrap_pairwise_mean(
    ordinal_pairwise_df["ordinal_pair_correct"],
    number_of_bootstraps=5000,
    random_seed=RANDOM_SEED + 4,
)


# ------------------------------------------------------------
# 7. Descriptive bilateral structure
# ------------------------------------------------------------

binary_concordant_patients = int(
    (~pairwise_df["binary_discordant"]).sum()
)

binary_discordant_patients = int(
    pairwise_df["binary_discordant"].sum()
)

ordinal_equal_patients = int(
    (~pairwise_df["ordinal_discordant"]).sum()
)

ordinal_unequal_patients = int(
    pairwise_df["ordinal_discordant"].sum()
)


# ------------------------------------------------------------
# 8. Save decomposition artifacts
# ------------------------------------------------------------

DECOMPOSITION_EYE_PATH = (
    OUTPUT_ROOT
    / "Validation_Between_Within_Decomposition_EyeLevel.csv"
)

PAIRWISE_AUDIT_PATH = (
    OUTPUT_ROOT
    / "Validation_WithinPatient_Pairwise_Audit.csv"
)

DECOMPOSITION_SUMMARY_PATH = (
    OUTPUT_ROOT
    / "Validation_Between_Within_Decomposition_Summary.json"
)

decomposition_df.to_csv(
    DECOMPOSITION_EYE_PATH,
    index=False,
)

pairwise_df.to_csv(
    PAIRWISE_AUDIT_PATH,
    index=False,
)

decomposition_summary = {
    "full_eye_level_auc": float(full_eye_auc),
    "full_eye_level_auc_95ci": [
        full_eye_auc_ci["lower"],
        full_eye_auc_ci["upper"],
    ],
    "between_patient_component_auc": float(
        between_patient_component_auc
    ),
    "between_patient_component_auc_95ci": [
        between_patient_auc_ci["lower"],
        between_patient_auc_ci["upper"],
    ],
    "binary_concordant_patients": (
        binary_concordant_patients
    ),
    "binary_discordant_patients": (
        binary_discordant_patients
    ),
    "binary_within_patient_centered_auc": (
        float(binary_within_centered_auc)
        if np.isfinite(binary_within_centered_auc)
        else None
    ),
    "binary_within_patient_centered_auc_95ci": [
        binary_within_auc_ci["lower"],
        binary_within_auc_ci["upper"],
    ],
    "binary_pairwise_accuracy": (
        binary_pairwise_result["estimate"]
    ),
    "binary_pairwise_accuracy_95ci": [
        binary_pairwise_result["lower"],
        binary_pairwise_result["upper"],
    ],
    "ordinal_equal_grade_patients": (
        ordinal_equal_patients
    ),
    "ordinal_unequal_grade_patients": (
        ordinal_unequal_patients
    ),
    "ordinal_pairwise_ranking_accuracy": (
        ordinal_pairwise_result["estimate"]
    ),
    "ordinal_pairwise_ranking_accuracy_95ci": [
        ordinal_pairwise_result["lower"],
        ordinal_pairwise_result["upper"],
    ],
}

with open(
    DECOMPOSITION_SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        decomposition_summary,
        file,
        indent=2,
    )


# ------------------------------------------------------------
# 9. Final report
# ------------------------------------------------------------

print(
    "================ BETWEEN/WITHIN DECOMPOSITION "
    "================"
)

print(
    "\nFull eye-level validation AUC: "
    f"{full_eye_auc:.4f} "
    f"[{full_eye_auc_ci['lower']:.4f}, "
    f"{full_eye_auc_ci['upper']:.4f}]"
)

print(
    "\nBetween-patient component AUC: "
    f"{between_patient_component_auc:.4f} "
    f"[{between_patient_auc_ci['lower']:.4f}, "
    f"{between_patient_auc_ci['upper']:.4f}]"
)

print("\nBinary bilateral structure:")
print(
    "  Concordant patients:",
    binary_concordant_patients,
)
print(
    "  Discordant patients:",
    binary_discordant_patients,
)

print(
    "\nBinary within-patient centered AUC: "
    f"{binary_within_centered_auc:.4f} "
    f"[{binary_within_auc_ci['lower']:.4f}, "
    f"{binary_within_auc_ci['upper']:.4f}]"
)

print(
    "Binary matched-pair accuracy: "
    f"{binary_pairwise_result['estimate']:.4f} "
    f"[{binary_pairwise_result['lower']:.4f}, "
    f"{binary_pairwise_result['upper']:.4f}] "
    f"(n={binary_pairwise_result['number_of_pairs']})"
)

print("\nOrdinal bilateral structure:")
print(
    "  Equal-grade patients:",
    ordinal_equal_patients,
)
print(
    "  Unequal-grade patients:",
    ordinal_unequal_patients,
)

print(
    "\nOrdinal within-patient ranking accuracy: "
    f"{ordinal_pairwise_result['estimate']:.4f} "
    f"[{ordinal_pairwise_result['lower']:.4f}, "
    f"{ordinal_pairwise_result['upper']:.4f}] "
    f"(n={ordinal_pairwise_result['number_of_pairs']})"
)

print("\nSaved:")
print(DECOMPOSITION_EYE_PATH)
print(PAIRWISE_AUDIT_PATH)
print(DECOMPOSITION_SUMMARY_PATH)

print(
    "\nBetween-patient versus within-patient "
    "decomposition passed."
)

================ BETWEEN/WITHIN DECOMPOSITION ================

Full eye-level validation AUC: 0.9392 [0.8968, 0.9738]

Between-patient component AUC: 0.9457 [0.9041, 0.9765]

Binary bilateral structure:
  Concordant patients: 90
  Discordant patients: 10

Binary within-patient centered AUC: 0.8400 [0.5200, 1.0000]
Binary matched-pair accuracy: 0.8000 [0.5000, 1.0000] (n=10)

Ordinal bilateral structure:
  Equal-grade patients: 55
  Unequal-grade patients: 45

Ordinal within-patient ranking accuracy: 0.6222 [0.4667, 0.7556] (n=45)

Saved:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Frozen_Representation_Baseline_v0.1/Validation_Between_Within_Decomposition_EyeLevel.csv
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Frozen_Representation_Baseline_v0.1/Validation_WithinPatient_Pairwise_Audit.csv
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Frozen_Representation_Bas

## 06. Own-eye versus fellow-eye locality test

In [ ]:
#@title 06.1 Own-eye versus fellow-eye locality test

import json
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score


# ------------------------------------------------------------
# 1. Construct own-eye and fellow-eye representations
# ------------------------------------------------------------

locality_meta = (
    eye_meta
    .reset_index(drop=True)
    .copy()
)

locality_meta["patient_id"] = (
    locality_meta["patient_id"]
    .astype(str)
)

locality_meta["eye_row"] = np.arange(
    len(locality_meta)
)

fellow_eye_rows = np.full(
    len(locality_meta),
    -1,
    dtype=int,
)

for (
    split_name,
    patient_id,
), patient_indices in locality_meta.groupby(
    ["split", "patient_id"],
    sort=False,
).groups.items():

    patient_indices = np.asarray(
        list(patient_indices),
        dtype=int,
    )

    if len(patient_indices) != 2:
        raise ValueError(
            f"Patient {patient_id} in {split_name} "
            f"has {len(patient_indices)} eyes."
        )

    if (
        locality_meta.loc[
            patient_indices,
            "eye_resolved",
        ].nunique()
        != 2
    ):
        raise ValueError(
            f"Patient {patient_id} does not have "
            "two distinct resolved eyes."
        )

    first_index, second_index = patient_indices

    fellow_eye_rows[first_index] = second_index
    fellow_eye_rows[second_index] = first_index


assert (fellow_eye_rows >= 0).all()

locality_meta["fellow_eye_row"] = fellow_eye_rows

locality_meta["fellow_eye_resolved"] = (
    locality_meta.loc[
        fellow_eye_rows,
        "eye_resolved",
    ]
    .to_numpy()
)

locality_meta["fellow_eye_DR_Level"] = (
    locality_meta.loc[
        fellow_eye_rows,
        "eye_DR_Level",
    ]
    .to_numpy(dtype=int)
)

locality_meta["fellow_eye_referable_dr"] = (
    locality_meta.loc[
        fellow_eye_rows,
        "referable_dr",
    ]
    .to_numpy(dtype=int)
)

own_eye_embeddings = eye_embeddings

fellow_eye_embeddings = eye_embeddings[
    fellow_eye_rows
]

assert own_eye_embeddings.shape == fellow_eye_embeddings.shape
assert own_eye_embeddings.shape == (800, embedding_dim)


# ------------------------------------------------------------
# 2. Official training and validation partitions
# ------------------------------------------------------------

training_mask = (
    locality_meta["split"]
    .eq("training")
    .to_numpy()
)

validation_mask = (
    locality_meta["split"]
    .eq("validation")
    .to_numpy()
)

target_labels_training = (
    locality_meta.loc[
        training_mask,
        "referable_dr",
    ]
    .to_numpy(dtype=int)
)

target_labels_validation = (
    locality_meta.loc[
        validation_mask,
        "referable_dr",
    ]
    .to_numpy(dtype=int)
)

training_patient_groups = (
    locality_meta.loc[
        training_mask,
        "patient_id",
    ]
    .to_numpy()
)

own_training_embeddings = (
    own_eye_embeddings[training_mask]
)

own_validation_embeddings = (
    own_eye_embeddings[validation_mask]
)

fellow_training_embeddings = (
    fellow_eye_embeddings[training_mask]
)

fellow_validation_embeddings = (
    fellow_eye_embeddings[validation_mask]
)

validation_locality_meta = (
    locality_meta.loc[
        validation_mask
    ]
    .reset_index(drop=True)
    .copy()
)


# ------------------------------------------------------------
# 3. Model specification
#
# Primary comparison uses the same C for own and fellow eye.
# This isolates representation locality from hyperparameter choice.
# ------------------------------------------------------------

def make_linear_probe(C_value):
    return Pipeline(
        steps=[
            (
                "standardize",
                StandardScaler(),
            ),
            (
                "classifier",
                LogisticRegression(
                    penalty="l2",
                    C=C_value,
                    solver="liblinear",
                    max_iter=10000,
                    random_state=RANDOM_SEED,
                ),
            ),
        ]
    )


MATCHED_C = best_C

grouped_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_SEED,
)


# ------------------------------------------------------------
# 4. Grouped out-of-fold prediction helper
# ------------------------------------------------------------

def grouped_oof_probabilities(
    feature_matrix,
    labels,
    patient_groups,
    C_value,
):
    predictions = np.full(
        len(labels),
        np.nan,
        dtype=float,
    )

    splitter = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOM_SEED,
    )

    for train_indices, test_indices in splitter.split(
        feature_matrix,
        labels,
        groups=patient_groups,
    ):
        model = make_linear_probe(C_value)

        model.fit(
            feature_matrix[train_indices],
            labels[train_indices],
        )

        predictions[test_indices] = (
            model.predict_proba(
                feature_matrix[test_indices]
            )[:, 1]
        )

    assert np.isfinite(predictions).all()

    return predictions


own_training_oof_probability = (
    grouped_oof_probabilities(
        feature_matrix=own_training_embeddings,
        labels=target_labels_training,
        patient_groups=training_patient_groups,
        C_value=MATCHED_C,
    )
)

fellow_training_oof_probability = (
    grouped_oof_probabilities(
        feature_matrix=fellow_training_embeddings,
        labels=target_labels_training,
        patient_groups=training_patient_groups,
        C_value=MATCHED_C,
    )
)


# ------------------------------------------------------------
# 5. Fit matched-C own-eye and fellow-eye probes
# ------------------------------------------------------------

own_eye_probe = make_linear_probe(
    MATCHED_C
)

fellow_eye_probe = make_linear_probe(
    MATCHED_C
)

own_eye_probe.fit(
    own_training_embeddings,
    target_labels_training,
)

fellow_eye_probe.fit(
    fellow_training_embeddings,
    target_labels_training,
)

own_validation_probability = (
    own_eye_probe.predict_proba(
        own_validation_embeddings
    )[:, 1]
)

fellow_validation_probability = (
    fellow_eye_probe.predict_proba(
        fellow_validation_embeddings
    )[:, 1]
)


# ------------------------------------------------------------
# 6. Primary held-out metrics
# ------------------------------------------------------------

own_training_oof_auc = roc_auc_score(
    target_labels_training,
    own_training_oof_probability,
)

fellow_training_oof_auc = roc_auc_score(
    target_labels_training,
    fellow_training_oof_probability,
)

own_validation_auc = roc_auc_score(
    target_labels_validation,
    own_validation_probability,
)

fellow_validation_auc = roc_auc_score(
    target_labels_validation,
    fellow_validation_probability,
)

own_validation_ap = average_precision_score(
    target_labels_validation,
    own_validation_probability,
)

fellow_validation_ap = average_precision_score(
    target_labels_validation,
    fellow_validation_probability,
)

observed_auc_difference = (
    own_validation_auc
    - fellow_validation_auc
)


# ------------------------------------------------------------
# 7. Patient-clustered paired bootstrap of AUC difference
# ------------------------------------------------------------

def paired_clustered_auc_bootstrap(
    metadata,
    labels,
    own_probabilities,
    fellow_probabilities,
    number_of_bootstraps=5000,
    random_seed=RANDOM_SEED,
):
    rng = np.random.default_rng(
        random_seed
    )

    metadata_patient_ids = (
        metadata["patient_id"]
        .astype(str)
        .to_numpy()
    )

    unique_patient_ids = np.unique(
        metadata_patient_ids
    )

    patient_to_indices = {
        patient_id: np.flatnonzero(
            metadata_patient_ids == patient_id
        )
        for patient_id in unique_patient_ids
    }

    own_auc_values = []
    fellow_auc_values = []
    difference_values = []

    for _ in range(number_of_bootstraps):
        sampled_patients = rng.choice(
            unique_patient_ids,
            size=len(unique_patient_ids),
            replace=True,
        )

        sampled_indices = np.concatenate(
            [
                patient_to_indices[patient_id]
                for patient_id in sampled_patients
            ]
        )

        sampled_labels = labels[
            sampled_indices
        ]

        if np.unique(sampled_labels).size < 2:
            continue

        sampled_own_auc = roc_auc_score(
            sampled_labels,
            own_probabilities[sampled_indices],
        )

        sampled_fellow_auc = roc_auc_score(
            sampled_labels,
            fellow_probabilities[sampled_indices],
        )

        own_auc_values.append(
            sampled_own_auc
        )

        fellow_auc_values.append(
            sampled_fellow_auc
        )

        difference_values.append(
            sampled_own_auc
            - sampled_fellow_auc
        )

    own_auc_values = np.asarray(
        own_auc_values,
        dtype=float,
    )

    fellow_auc_values = np.asarray(
        fellow_auc_values,
        dtype=float,
    )

    difference_values = np.asarray(
        difference_values,
        dtype=float,
    )

    return {
        "own_auc_lower": float(
            np.percentile(
                own_auc_values,
                2.5,
            )
        ),
        "own_auc_upper": float(
            np.percentile(
                own_auc_values,
                97.5,
            )
        ),
        "fellow_auc_lower": float(
            np.percentile(
                fellow_auc_values,
                2.5,
            )
        ),
        "fellow_auc_upper": float(
            np.percentile(
                fellow_auc_values,
                97.5,
            )
        ),
        "difference_lower": float(
            np.percentile(
                difference_values,
                2.5,
            )
        ),
        "difference_upper": float(
            np.percentile(
                difference_values,
                97.5,
            )
        ),
        "probability_own_greater": float(
            np.mean(
                difference_values > 0
            )
        ),
        "valid_bootstraps": int(
            len(difference_values)
        ),
    }


overall_bootstrap = (
    paired_clustered_auc_bootstrap(
        metadata=validation_locality_meta,
        labels=target_labels_validation,
        own_probabilities=(
            own_validation_probability
        ),
        fellow_probabilities=(
            fellow_validation_probability
        ),
        number_of_bootstraps=5000,
        random_seed=RANDOM_SEED + 10,
    )
)


# ------------------------------------------------------------
# 8. Binary-discordant patient subset
# ------------------------------------------------------------

validation_locality_meta[
    "target_label"
] = target_labels_validation

validation_locality_meta[
    "own_probability"
] = own_validation_probability

validation_locality_meta[
    "fellow_probability"
] = fellow_validation_probability

validation_locality_meta[
    "patient_referable_count"
] = (
    validation_locality_meta
    .groupby("patient_id")[
        "target_label"
    ]
    .transform("sum")
    .astype(int)
)

discordant_validation = (
    validation_locality_meta.loc[
        validation_locality_meta[
            "patient_referable_count"
        ].eq(1)
    ]
    .reset_index(drop=True)
    .copy()
)

discordant_labels = (
    discordant_validation[
        "target_label"
    ]
    .to_numpy(dtype=int)
)

discordant_own_probability = (
    discordant_validation[
        "own_probability"
    ]
    .to_numpy(dtype=float)
)

discordant_fellow_probability = (
    discordant_validation[
        "fellow_probability"
    ]
    .to_numpy(dtype=float)
)

if (
    len(discordant_validation) > 0
    and np.unique(discordant_labels).size == 2
):
    discordant_own_auc = roc_auc_score(
        discordant_labels,
        discordant_own_probability,
    )

    discordant_fellow_auc = roc_auc_score(
        discordant_labels,
        discordant_fellow_probability,
    )

    discordant_auc_difference = (
        discordant_own_auc
        - discordant_fellow_auc
    )

    discordant_bootstrap = (
        paired_clustered_auc_bootstrap(
            metadata=discordant_validation,
            labels=discordant_labels,
            own_probabilities=(
                discordant_own_probability
            ),
            fellow_probabilities=(
                discordant_fellow_probability
            ),
            number_of_bootstraps=5000,
            random_seed=RANDOM_SEED + 11,
        )
    )

else:
    discordant_own_auc = np.nan
    discordant_fellow_auc = np.nan
    discordant_auc_difference = np.nan

    discordant_bootstrap = {
        "difference_lower": np.nan,
        "difference_upper": np.nan,
        "probability_own_greater": np.nan,
        "valid_bootstraps": 0,
    }


# ------------------------------------------------------------
# 9. Ordinal within-patient ranking:
# own-eye scores versus fellow-eye scores
# ------------------------------------------------------------

ordinal_locality_records = []

for patient_id, patient_group in (
    validation_locality_meta.groupby(
        "patient_id",
        sort=False,
    )
):
    patient_group = patient_group.reset_index(
        drop=True
    )

    grades = (
        patient_group["eye_DR_Level"]
        .to_numpy(dtype=int)
    )

    if grades[0] == grades[1]:
        continue

    higher_grade_index = int(
        np.argmax(grades)
    )

    lower_grade_index = (
        1 - higher_grade_index
    )

    own_scores = (
        patient_group["own_probability"]
        .to_numpy(dtype=float)
    )

    fellow_scores = (
        patient_group["fellow_probability"]
        .to_numpy(dtype=float)
    )

    own_margin = (
        own_scores[higher_grade_index]
        - own_scores[lower_grade_index]
    )

    fellow_margin = (
        fellow_scores[higher_grade_index]
        - fellow_scores[lower_grade_index]
    )

    ordinal_locality_records.append(
        {
            "patient_id": str(patient_id),
            "lower_grade": int(
                grades[lower_grade_index]
            ),
            "higher_grade": int(
                grades[higher_grade_index]
            ),
            "own_margin": float(
                own_margin
            ),
            "fellow_margin": float(
                fellow_margin
            ),
            "own_correct": (
                1.0 if own_margin > 0
                else 0.0 if own_margin < 0
                else 0.5
            ),
            "fellow_correct": (
                1.0 if fellow_margin > 0
                else 0.0 if fellow_margin < 0
                else 0.5
            ),
        }
    )


ordinal_locality_df = pd.DataFrame(
    ordinal_locality_records
)

own_ordinal_result = bootstrap_pairwise_mean(
    ordinal_locality_df[
        "own_correct"
    ],
    number_of_bootstraps=5000,
    random_seed=RANDOM_SEED + 12,
)

fellow_ordinal_result = bootstrap_pairwise_mean(
    ordinal_locality_df[
        "fellow_correct"
    ],
    number_of_bootstraps=5000,
    random_seed=RANDOM_SEED + 13,
)

ordinal_accuracy_difference_values = (
    ordinal_locality_df["own_correct"]
    - ordinal_locality_df["fellow_correct"]
).to_numpy(dtype=float)

ordinal_difference_result = (
    bootstrap_pairwise_mean(
        ordinal_accuracy_difference_values,
        number_of_bootstraps=5000,
        random_seed=RANDOM_SEED + 14,
    )
)


# ------------------------------------------------------------
# 10. Save artifacts
# ------------------------------------------------------------

LOCALITY_PREDICTIONS_PATH = (
    OUTPUT_ROOT
    / "Validation_OwnEye_vs_FellowEye_Predictions.csv"
)

LOCALITY_ORDINAL_PATH = (
    OUTPUT_ROOT
    / "Validation_OwnEye_vs_FellowEye_Ordinal_Audit.csv"
)

LOCALITY_SUMMARY_PATH = (
    OUTPUT_ROOT
    / "Validation_OwnEye_vs_FellowEye_Summary.json"
)

validation_locality_meta.to_csv(
    LOCALITY_PREDICTIONS_PATH,
    index=False,
)

ordinal_locality_df.to_csv(
    LOCALITY_ORDINAL_PATH,
    index=False,
)

locality_summary = {
    "matched_regularization_C": float(
        MATCHED_C
    ),
    "training_own_eye_grouped_oof_auc": float(
        own_training_oof_auc
    ),
    "training_fellow_eye_grouped_oof_auc": float(
        fellow_training_oof_auc
    ),
    "validation_own_eye_auc": float(
        own_validation_auc
    ),
    "validation_own_eye_average_precision": float(
        own_validation_ap
    ),
    "validation_fellow_eye_auc": float(
        fellow_validation_auc
    ),
    "validation_fellow_eye_average_precision": float(
        fellow_validation_ap
    ),
    "validation_own_minus_fellow_auc": float(
        observed_auc_difference
    ),
    "validation_own_minus_fellow_auc_95ci": [
        overall_bootstrap[
            "difference_lower"
        ],
        overall_bootstrap[
            "difference_upper"
        ],
    ],
    "binary_discordant_patients": int(
        discordant_validation[
            "patient_id"
        ].nunique()
    ),
    "discordant_own_eye_auc": (
        float(discordant_own_auc)
        if np.isfinite(discordant_own_auc)
        else None
    ),
    "discordant_fellow_eye_auc": (
        float(discordant_fellow_auc)
        if np.isfinite(discordant_fellow_auc)
        else None
    ),
    "discordant_own_minus_fellow_auc": (
        float(discordant_auc_difference)
        if np.isfinite(
            discordant_auc_difference
        )
        else None
    ),
    "ordinal_unequal_grade_patients": int(
        len(ordinal_locality_df)
    ),
    "ordinal_own_eye_ranking_accuracy": (
        own_ordinal_result["estimate"]
    ),
    "ordinal_fellow_eye_ranking_accuracy": (
        fellow_ordinal_result["estimate"]
    ),
    "ordinal_own_minus_fellow_accuracy": (
        ordinal_difference_result["estimate"]
    ),
    "ordinal_own_minus_fellow_accuracy_95ci": [
        ordinal_difference_result[
            "lower"
        ],
        ordinal_difference_result[
            "upper"
        ],
    ],
}

with open(
    LOCALITY_SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        locality_summary,
        file,
        indent=2,
    )


# ------------------------------------------------------------
# 11. Final report
# ------------------------------------------------------------

print(
    "================ OWN-EYE VS FELLOW-EYE "
    "LOCALITY TEST ================"
)

print(
    "\nMatched regularization C:",
    MATCHED_C,
)

print(
    "\nTraining grouped OOF AUC:"
)

print(
    "  Own eye:   "
    f"{own_training_oof_auc:.4f}"
)

print(
    "  Fellow eye:"
    f" {fellow_training_oof_auc:.4f}"
)

print(
    "\nOfficial validation:"
)

print(
    "  Own-eye AUC:    "
    f"{own_validation_auc:.4f} "
    f"[{overall_bootstrap['own_auc_lower']:.4f}, "
    f"{overall_bootstrap['own_auc_upper']:.4f}]"
)

print(
    "  Fellow-eye AUC: "
    f"{fellow_validation_auc:.4f} "
    f"[{overall_bootstrap['fellow_auc_lower']:.4f}, "
    f"{overall_bootstrap['fellow_auc_upper']:.4f}]"
)

print(
    "  Own − fellow AUC difference: "
    f"{observed_auc_difference:.4f} "
    f"[{overall_bootstrap['difference_lower']:.4f}, "
    f"{overall_bootstrap['difference_upper']:.4f}]"
)

print(
    "  Bootstrap P(own > fellow): "
    f"{overall_bootstrap['probability_own_greater']:.4f}"
)

print(
    "\nBinary-discordant subset:"
)

print(
    "  Patients:",
    discordant_validation[
        "patient_id"
    ].nunique(),
)

print(
    "  Own-eye AUC:    "
    f"{discordant_own_auc:.4f}"
)

print(
    "  Fellow-eye AUC: "
    f"{discordant_fellow_auc:.4f}"
)

print(
    "  Own − fellow AUC difference: "
    f"{discordant_auc_difference:.4f} "
    f"[{discordant_bootstrap['difference_lower']:.4f}, "
    f"{discordant_bootstrap['difference_upper']:.4f}]"
)

print(
    "\nOrdinal unequal-grade subset:"
)

print(
    "  Patients:",
    len(ordinal_locality_df),
)

print(
    "  Own-eye ranking accuracy: "
    f"{own_ordinal_result['estimate']:.4f} "
    f"[{own_ordinal_result['lower']:.4f}, "
    f"{own_ordinal_result['upper']:.4f}]"
)

print(
    "  Fellow-eye ranking accuracy: "
    f"{fellow_ordinal_result['estimate']:.4f} "
    f"[{fellow_ordinal_result['lower']:.4f}, "
    f"{fellow_ordinal_result['upper']:.4f}]"
)

print(
    "  Own − fellow accuracy difference: "
    f"{ordinal_difference_result['estimate']:.4f} "
    f"[{ordinal_difference_result['lower']:.4f}, "
    f"{ordinal_difference_result['upper']:.4f}]"
)

print("\nSaved:")
print(LOCALITY_PREDICTIONS_PATH)
print(LOCALITY_ORDINAL_PATH)
print(LOCALITY_SUMMARY_PATH)

print(
    "\nOwn-eye versus fellow-eye locality "
    "test passed."
)

================ OWN-EYE VS FELLOW-EYE LOCALITY TEST ================

Matched regularization C: 0.003

Training grouped OOF AUC:
  Own eye:   0.9143
  Fellow eye: 0.8873

Official validation:
  Own-eye AUC:    0.9392 [0.8947, 0.9751]
  Fellow-eye AUC: 0.9310 [0.8874, 0.9661]
  Own − fellow AUC difference: 0.0082 [-0.0270, 0.0399]
  Bootstrap P(own > fellow): 0.7100

Binary-discordant subset:
  Patients: 10
  Own-eye AUC:    0.7300
  Fellow-eye AUC: 0.3600
  Own − fellow AUC difference: 0.3700 [0.0300, 0.7100]

Ordinal unequal-grade subset:
  Patients: 45
  Own-eye ranking accuracy: 0.6222 [0.4889, 0.7556]
  Fellow-eye ranking accuracy: 0.3778 [0.2444, 0.5333]
  Own − fellow accuracy difference: 0.2444 [-0.0222, 0.4889]

Saved:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Frozen_Representation_Baseline_v0.1/Validation_OwnEye_vs_FellowEye_Predictions.csv
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Froze

## 07. Within-patient ordinal severity ranking

In [ ]:
#@title 07.1 Symmetric paired-difference probe for local severity

import json
import joblib
import numpy as np
import pandas as pd

from scipy.stats import binomtest

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
)


# ------------------------------------------------------------
# 1. Prepare one row per eye
# ------------------------------------------------------------

paired_eye_meta = (
    eye_meta
    .reset_index(drop=True)
    .copy()
)

paired_eye_meta["eye_row"] = np.arange(
    len(paired_eye_meta)
)

paired_eye_meta["patient_id"] = (
    paired_eye_meta["patient_id"]
    .astype(str)
)


def normalise_eye_name(value):
    value = str(value).strip().lower()

    if value.startswith("l"):
        return "L"

    if value.startswith("r"):
        return "R"

    raise ValueError(
        f"Unrecognised eye name: {value}"
    )


paired_eye_meta["canonical_eye"] = (
    paired_eye_meta["eye_resolved"]
    .map(normalise_eye_name)
)

assert paired_eye_meta[
    "canonical_eye"
].isin(["L", "R"]).all()


# ------------------------------------------------------------
# 2. Construct patient-paired embedding differences
#
# Delta X = left-eye embedding - right-eye embedding
# Label = 1 when left eye has the higher DR grade
# ------------------------------------------------------------

pair_records = []
pair_embedding_differences = []

for (
    split_name,
    patient_id,
), patient_group in paired_eye_meta.groupby(
    ["split", "patient_id"],
    sort=False,
):
    patient_group = (
        patient_group
        .reset_index(drop=True)
    )

    if len(patient_group) != 2:
        raise ValueError(
            f"{split_name}, patient {patient_id}: "
            f"expected two eyes, found "
            f"{len(patient_group)}"
        )

    left_group = patient_group.loc[
        patient_group["canonical_eye"].eq("L")
    ]

    right_group = patient_group.loc[
        patient_group["canonical_eye"].eq("R")
    ]

    if (
        len(left_group) != 1
        or len(right_group) != 1
    ):
        raise ValueError(
            f"{split_name}, patient {patient_id}: "
            "could not uniquely resolve left and right eyes."
        )

    left_row = left_group.iloc[0]
    right_row = right_group.iloc[0]

    left_embedding_row = int(
        left_row["eye_row"]
    )

    right_embedding_row = int(
        right_row["eye_row"]
    )

    left_grade = int(
        left_row["eye_DR_Level"]
    )

    right_grade = int(
        right_row["eye_DR_Level"]
    )

    grade_difference = (
        left_grade - right_grade
    )

    # Equal-grade pairs contain no ordinal direction label.
    if grade_difference == 0:
        continue

    left_referable = int(
        left_grade >= 2
    )

    right_referable = int(
        right_grade >= 2
    )

    embedding_difference = (
        eye_embeddings[left_embedding_row]
        - eye_embeddings[right_embedding_row]
    )

    pair_records.append(
        {
            "split": split_name,
            "patient_id": str(patient_id),
            "left_grade": left_grade,
            "right_grade": right_grade,
            "absolute_grade_gap": abs(
                grade_difference
            ),
            "left_higher_grade": int(
                grade_difference > 0
            ),
            "left_referable": left_referable,
            "right_referable": right_referable,
            "binary_referable_discordant": int(
                left_referable
                != right_referable
            ),
            "left_source": str(
                left_row["Source"]
            ),
            "right_source": str(
                right_row["Source"]
            ),
        }
    )

    pair_embedding_differences.append(
        embedding_difference
    )


ordinal_pair_meta = pd.DataFrame(
    pair_records
)

ordinal_pair_embeddings = np.vstack(
    pair_embedding_differences
).astype(np.float32)

assert len(ordinal_pair_meta) == len(
    ordinal_pair_embeddings
)

assert np.isfinite(
    ordinal_pair_embeddings
).all()


# ------------------------------------------------------------
# 3. Official training and validation partitions
# ------------------------------------------------------------

training_pair_mask = (
    ordinal_pair_meta["split"]
    .eq("training")
    .to_numpy()
)

validation_pair_mask = (
    ordinal_pair_meta["split"]
    .eq("validation")
    .to_numpy()
)

X_pair_train = ordinal_pair_embeddings[
    training_pair_mask
]

X_pair_validation = ordinal_pair_embeddings[
    validation_pair_mask
]

y_pair_train = (
    ordinal_pair_meta.loc[
        training_pair_mask,
        "left_higher_grade",
    ]
    .to_numpy(dtype=int)
)

y_pair_validation = (
    ordinal_pair_meta.loc[
        validation_pair_mask,
        "left_higher_grade",
    ]
    .to_numpy(dtype=int)
)

training_pair_meta = (
    ordinal_pair_meta.loc[
        training_pair_mask
    ]
    .reset_index(drop=True)
    .copy()
)

validation_pair_meta = (
    ordinal_pair_meta.loc[
        validation_pair_mask
    ]
    .reset_index(drop=True)
    .copy()
)


print(
    "================ PAIRED ORDINAL DATA AUDIT "
    "================"
)

print(
    "\nUnequal-grade training patients:",
    len(training_pair_meta),
)

print(
    "Unequal-grade validation patients:",
    len(validation_pair_meta),
)

print(
    "\nValidation grade-gap distribution:"
)

print(
    validation_pair_meta[
        "absolute_grade_gap"
    ]
    .value_counts()
    .sort_index()
)

print(
    "\nValidation binary-discordant patients:",
    int(
        validation_pair_meta[
            "binary_referable_discordant"
        ].sum()
    ),
)

print(
    "\nLeft eye is more severe — prevalence:"
)

print(
    ordinal_pair_meta.groupby(
        "split"
    )["left_higher_grade"].mean()
)

assert len(validation_pair_meta) == 45


# ------------------------------------------------------------
# 4. Symmetric augmentation
#
# For every Delta X with label y, add -Delta X with label 1-y.
# This prevents the model from learning a left/right-side bias.
# ------------------------------------------------------------

def symmetrically_augment(
    feature_matrix,
    labels,
):
    augmented_features = np.concatenate(
        [
            feature_matrix,
            -feature_matrix,
        ],
        axis=0,
    )

    augmented_labels = np.concatenate(
        [
            labels,
            1 - labels,
        ],
        axis=0,
    )

    return (
        augmented_features,
        augmented_labels,
    )


def make_paired_probe(C_value):
    return Pipeline(
        steps=[
            (
                "scale",
                StandardScaler(
                    with_mean=False
                ),
            ),
            (
                "classifier",
                LogisticRegression(
                    penalty="l2",
                    C=C_value,
                    solver="liblinear",
                    fit_intercept=False,
                    max_iter=10000,
                    random_state=RANDOM_SEED,
                ),
            ),
        ]
    )


# ------------------------------------------------------------
# 5. Select regularisation using training patients only
# ------------------------------------------------------------

PAIR_C_GRID = [
    1e-5,
    3e-5,
    1e-4,
    3e-4,
    1e-3,
    3e-3,
    1e-2,
    3e-2,
    1e-1,
]

pair_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_SEED,
)

pair_cv_rows = []

for C_value in PAIR_C_GRID:
    fold_accuracies = []
    fold_balanced_accuracies = []
    fold_aucs = []

    for (
        fold_train_indices,
        fold_test_indices,
    ) in pair_cv.split(
        X_pair_train,
        y_pair_train,
    ):
        (
            augmented_fold_features,
            augmented_fold_labels,
        ) = symmetrically_augment(
            X_pair_train[
                fold_train_indices
            ],
            y_pair_train[
                fold_train_indices
            ],
        )

        fold_model = make_paired_probe(
            C_value
        )

        fold_model.fit(
            augmented_fold_features,
            augmented_fold_labels,
        )

        fold_probabilities = (
            fold_model.predict_proba(
                X_pair_train[
                    fold_test_indices
                ]
            )[:, 1]
        )

        fold_predictions = (
            fold_probabilities >= 0.5
        ).astype(int)

        fold_labels = y_pair_train[
            fold_test_indices
        ]

        fold_accuracies.append(
            accuracy_score(
                fold_labels,
                fold_predictions,
            )
        )

        fold_balanced_accuracies.append(
            balanced_accuracy_score(
                fold_labels,
                fold_predictions,
            )
        )

        if np.unique(
            fold_labels
        ).size == 2:
            fold_aucs.append(
                roc_auc_score(
                    fold_labels,
                    fold_probabilities,
                )
            )

    pair_cv_rows.append(
        {
            "C": C_value,
            "mean_cv_accuracy": float(
                np.mean(
                    fold_accuracies
                )
            ),
            "sd_cv_accuracy": float(
                np.std(
                    fold_accuracies,
                    ddof=1,
                )
            ),
            "mean_cv_balanced_accuracy": float(
                np.mean(
                    fold_balanced_accuracies
                )
            ),
            "sd_cv_balanced_accuracy": float(
                np.std(
                    fold_balanced_accuracies,
                    ddof=1,
                )
            ),
            "mean_cv_auc": float(
                np.mean(
                    fold_aucs
                )
            ),
        }
    )


pair_cv_results = pd.DataFrame(
    pair_cv_rows
)

pair_cv_results = (
    pair_cv_results
    .sort_values(
        by=[
            "mean_cv_balanced_accuracy",
            "mean_cv_accuracy",
            "C",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

best_pair_C = float(
    pair_cv_results.loc[0, "C"]
)


print(
    "\n================ PAIRED-PROBE CV "
    "================"
)

display(pair_cv_results)

print(
    "Selected paired-probe C:",
    best_pair_C,
)


# ------------------------------------------------------------
# 6. Train the final symmetric paired-difference model
# ------------------------------------------------------------

(
    augmented_training_features,
    augmented_training_labels,
) = symmetrically_augment(
    X_pair_train,
    y_pair_train,
)

paired_probe = make_paired_probe(
    best_pair_C
)

paired_probe.fit(
    augmented_training_features,
    augmented_training_labels,
)


# ------------------------------------------------------------
# 7. Evaluate on official held-out validation patients
# ------------------------------------------------------------

validation_pair_probability = (
    paired_probe.predict_proba(
        X_pair_validation
    )[:, 1]
)

validation_pair_prediction = (
    validation_pair_probability >= 0.5
).astype(int)

validation_pair_decision = (
    paired_probe.decision_function(
        X_pair_validation
    )
)

validation_pair_correct = (
    validation_pair_prediction
    == y_pair_validation
).astype(float)

validation_pair_accuracy = accuracy_score(
    y_pair_validation,
    validation_pair_prediction,
)

validation_pair_balanced_accuracy = (
    balanced_accuracy_score(
        y_pair_validation,
        validation_pair_prediction,
    )
)

validation_pair_auc = roc_auc_score(
    y_pair_validation,
    validation_pair_probability,
)

correct_pairs = int(
    validation_pair_correct.sum()
)

number_of_validation_pairs = int(
    len(validation_pair_correct)
)

exact_binomial_result = binomtest(
    k=correct_pairs,
    n=number_of_validation_pairs,
    p=0.5,
    alternative="greater",
)

validation_accuracy_ci = (
    bootstrap_pairwise_mean(
        validation_pair_correct,
        number_of_bootstraps=10000,
        random_seed=RANDOM_SEED + 20,
    )
)

true_direction_sign = (
    2 * y_pair_validation - 1
)

validation_true_signed_margin = (
    validation_pair_decision
    * true_direction_sign
)


# ------------------------------------------------------------
# 8. Add validation predictions to the audit table
# ------------------------------------------------------------

validation_pair_meta[
    "probability_left_higher"
] = validation_pair_probability

validation_pair_meta[
    "predicted_left_higher"
] = validation_pair_prediction

validation_pair_meta[
    "correct"
] = validation_pair_correct

validation_pair_meta[
    "true_signed_margin"
] = validation_true_signed_margin


# ------------------------------------------------------------
# 9. Grade-gap and binary-discordant subgroups
# ------------------------------------------------------------

subgroup_rows = []

for (
    subgroup_name,
    subgroup_mask,
) in [
    (
        "grade_gap_1",
        validation_pair_meta[
            "absolute_grade_gap"
        ].eq(1),
    ),
    (
        "grade_gap_2_or_more",
        validation_pair_meta[
            "absolute_grade_gap"
        ].ge(2),
    ),
    (
        "binary_referable_discordant",
        validation_pair_meta[
            "binary_referable_discordant"
        ].eq(1),
    ),
]:
    subgroup = (
        validation_pair_meta.loc[
            subgroup_mask
        ]
        .reset_index(drop=True)
    )

    if len(subgroup) == 0:
        continue

    subgroup_correct = (
        subgroup["correct"]
        .to_numpy(dtype=float)
    )

    subgroup_ci = bootstrap_pairwise_mean(
        subgroup_correct,
        number_of_bootstraps=10000,
        random_seed=(
            RANDOM_SEED
            + 30
            + len(subgroup_rows)
        ),
    )

    subgroup_rows.append(
        {
            "subgroup": subgroup_name,
            "patients": int(
                len(subgroup)
            ),
            "accuracy": float(
                subgroup_correct.mean()
            ),
            "accuracy_ci_lower": (
                subgroup_ci["lower"]
            ),
            "accuracy_ci_upper": (
                subgroup_ci["upper"]
            ),
        }
    )


paired_subgroup_results = pd.DataFrame(
    subgroup_rows
)


# ------------------------------------------------------------
# 10. Save artifacts
# ------------------------------------------------------------

PAIRED_CV_PATH = (
    OUTPUT_ROOT
    / "Ordinal_PairedDifference_Probe_CV.csv"
)

PAIRED_VALIDATION_PATH = (
    OUTPUT_ROOT
    / "Validation_Ordinal_PairedDifference_Predictions.csv"
)

PAIRED_MODEL_PATH = (
    OUTPUT_ROOT
    / "Ordinal_PairedDifference_Probe.joblib"
)

PAIRED_SUMMARY_PATH = (
    OUTPUT_ROOT
    / "Validation_Ordinal_PairedDifference_Summary.json"
)

pair_cv_results.to_csv(
    PAIRED_CV_PATH,
    index=False,
)

validation_pair_meta.to_csv(
    PAIRED_VALIDATION_PATH,
    index=False,
)

joblib.dump(
    paired_probe,
    PAIRED_MODEL_PATH,
)

paired_summary = {
    "representation": (
        "ResNet-50 ImageNet-1K V2 frozen"
    ),
    "analysis": (
        "symmetric left-minus-right "
        "paired-difference probe"
    ),
    "training_unequal_grade_patients": int(
        len(training_pair_meta)
    ),
    "validation_unequal_grade_patients": int(
        len(validation_pair_meta)
    ),
    "selected_C": best_pair_C,
    "validation_accuracy": float(
        validation_pair_accuracy
    ),
    "validation_accuracy_95ci": [
        validation_accuracy_ci["lower"],
        validation_accuracy_ci["upper"],
    ],
    "validation_balanced_accuracy": float(
        validation_pair_balanced_accuracy
    ),
    "validation_auc": float(
        validation_pair_auc
    ),
    "correct_pairs": correct_pairs,
    "exact_one_sided_binomial_p": float(
        exact_binomial_result.pvalue
    ),
    "mean_true_signed_margin": float(
        validation_true_signed_margin.mean()
    ),
}

with open(
    PAIRED_SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        paired_summary,
        file,
        indent=2,
    )


# ------------------------------------------------------------
# 11. Final report
# ------------------------------------------------------------

print(
    "\n================ PAIRED-DIFFERENCE "
    "VALIDATION RESULT ================"
)

print(
    "Training unequal-grade patients:",
    len(training_pair_meta),
)

print(
    "Validation unequal-grade patients:",
    len(validation_pair_meta),
)

print(
    "Selected C:",
    best_pair_C,
)

print(
    "\nValidation pairwise accuracy: "
    f"{validation_pair_accuracy:.4f} "
    f"[{validation_accuracy_ci['lower']:.4f}, "
    f"{validation_accuracy_ci['upper']:.4f}]"
)

print(
    "Validation balanced accuracy: "
    f"{validation_pair_balanced_accuracy:.4f}"
)

print(
    "Validation directional AUC: "
    f"{validation_pair_auc:.4f}"
)

print(
    "Correct pairs: "
    f"{correct_pairs}/"
    f"{number_of_validation_pairs}"
)

print(
    "Exact one-sided binomial p-value: "
    f"{exact_binomial_result.pvalue:.6f}"
)

print(
    "Mean true-signed decision margin: "
    f"{validation_true_signed_margin.mean():.4f}"
)

print(
    "\nSubgroup results:"
)

display(paired_subgroup_results)

print("\nSaved:")
print(PAIRED_CV_PATH)
print(PAIRED_VALIDATION_PATH)
print(PAIRED_MODEL_PATH)
print(PAIRED_SUMMARY_PATH)

print(
    "\nSymmetric paired-difference locality "
    "probe passed."
)

================ PAIRED ORDINAL DATA AUDIT ================

Unequal-grade training patients: 134
Unequal-grade validation patients: 45

Validation grade-gap distribution:
absolute_grade_gap
1    32
2    10
3     1
4     2
Name: count, dtype: int64

Validation binary-discordant patients: 10

Left eye is more severe — prevalence:
split
training      0.559701
validation    0.422222
Name: left_higher_grade, dtype: float64

================ PAIRED-PROBE CV ================


,C,mean_cv_accuracy,sd_cv_accuracy,mean_cv_balanced_accuracy,sd_cv_balanced_accuracy,mean_cv_auc
0,0.00100,0.656980,0.087130,0.659394,0.090467,0.718384
1,0.00300,0.649573,0.059761,0.651061,0.061617,0.720707
2,0.01000,0.649573,0.059761,0.649394,0.067410,0.715859
3,0.03000,0.642165,0.059407,0.641061,0.066014,0.704444
4,0.10000,0.642165,0.059407,0.641061,0.066014,0.706566
5,0.00030,0.634758,0.082336,0.639394,0.086369,0.709394
6,0.00010,0.612536,0.090341,0.617727,0.089149,0.698283
7,0.00001,0.604843,0.074508,0.611970,0.071310,0.691717
8,0.00003,0.597436,0.082089,0.603636,0.079850,0.693939


Selected paired-probe C: 0.001

================ PAIRED-DIFFERENCE VALIDATION RESULT ================
Training unequal-grade patients: 134
Validation unequal-grade patients: 45
Selected C: 0.001

Validation pairwise accuracy: 0.6889 [0.5556, 0.8222]
Validation balanced accuracy: 0.6883
Validation directional AUC: 0.7530
Correct pairs: 31/45
Exact one-sided binomial p-value: 0.008047
Mean true-signed decision margin: 0.2933

Subgroup results:


,subgroup,patients,accuracy,accuracy_ci_lower,accuracy_ci_upper
0,grade_gap_1,32,0.656250,0.500000,0.8125
1,grade_gap_2_or_more,13,0.769231,0.538462,1.0000
2,binary_referable_discordant,10,0.800000,0.500000,1.0000



Saved:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Frozen_Representation_Baseline_v0.1/Ordinal_PairedDifference_Probe_CV.csv
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Frozen_Representation_Baseline_v0.1/Validation_Ordinal_PairedDifference_Predictions.csv
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Frozen_Representation_Baseline_v0.1/Ordinal_PairedDifference_Probe.joblib
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Frozen_Representation_Baseline_v0.1/Validation_Ordinal_PairedDifference_Summary.json

Symmetric paired-difference locality probe passed.


## 08. Results, decision gate and reproducibility outputs

In [ ]:
#@title 08.1 Consolidate evidence and issue the baseline decision

from pathlib import Path
from scipy.stats import binomtest

import hashlib
import json
import platform
import sklearn
import torchvision
import numpy as np
import pandas as pd
import torch


# ------------------------------------------------------------
# 1. Exact confidence interval for the primary locality result
# ------------------------------------------------------------

exact_two_sided_result = binomtest(
    k=correct_pairs,
    n=number_of_validation_pairs,
    p=0.5,
    alternative="two-sided",
)

exact_accuracy_ci = exact_two_sided_result.proportion_ci(
    confidence_level=0.95,
    method="exact",
)

exact_accuracy_ci_lower = float(exact_accuracy_ci.low)
exact_accuracy_ci_upper = float(exact_accuracy_ci.high)


# ------------------------------------------------------------
# 2. Retrieve the major acquisition strata
# ------------------------------------------------------------

def get_stratum_row(
    metrics_dataframe,
    stratum_type,
    stratum_name,
):
    matches = metrics_dataframe.loc[
        metrics_dataframe["stratum_type"].eq(stratum_type)
        & metrics_dataframe["stratum"].eq(stratum_name)
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected one row for "
            f"{stratum_type}={stratum_name}, "
            f"found {len(matches)}"
        )

    return matches.iloc[0]


nicheng_result = get_stratum_row(
    stratified_metrics,
    "Source",
    "Nicheng",
)

resolution_1736_result = get_stratum_row(
    stratified_metrics,
    "resolution",
    "1736x1824",
)

resolution_1976_result = get_stratum_row(
    stratified_metrics,
    "resolution",
    "1976x1984",
)


# ------------------------------------------------------------
# 3. Consolidated evidence table
# ------------------------------------------------------------

evidence_rows = [
    {
        "evidence_domain": "Diagnostic predictability",
        "analysis": "Official held-out eye-level linear probe",
        "sample": "100 patients / 200 eyes",
        "metric": "AUC",
        "estimate": validation_auc,
        "ci_lower": validation_auc_ci["lower"],
        "ci_upper": validation_auc_ci["upper"],
        "p_value": np.nan,
        "interpretation": (
            "Strong held-out diagnostic predictability"
        ),
    },
    {
        "evidence_domain": "Acquisition control",
        "analysis": "Nicheng source only",
        "sample": (
            f"{int(nicheng_result['patients'])} patients / "
            f"{int(nicheng_result['eyes'])} eyes"
        ),
        "metric": "AUC",
        "estimate": float(nicheng_result["auc"]),
        "ci_lower": float(nicheng_result["auc_ci_lower"]),
        "ci_upper": float(nicheng_result["auc_ci_upper"]),
        "p_value": np.nan,
        "interpretation": (
            "Strong performance remains within the main source"
        ),
    },
    {
        "evidence_domain": "Acquisition control",
        "analysis": "Fixed resolution 1736x1824",
        "sample": (
            f"{int(resolution_1736_result['patients'])} patients / "
            f"{int(resolution_1736_result['eyes'])} eyes"
        ),
        "metric": "AUC",
        "estimate": float(resolution_1736_result["auc"]),
        "ci_lower": float(
            resolution_1736_result["auc_ci_lower"]
        ),
        "ci_upper": float(
            resolution_1736_result["auc_ci_upper"]
        ),
        "p_value": np.nan,
        "interpretation": (
            "Strong performance remains at fixed resolution"
        ),
    },
    {
        "evidence_domain": "Acquisition control",
        "analysis": "Fixed resolution 1976x1984",
        "sample": (
            f"{int(resolution_1976_result['patients'])} patients / "
            f"{int(resolution_1976_result['eyes'])} eyes"
        ),
        "metric": "AUC",
        "estimate": float(resolution_1976_result["auc"]),
        "ci_lower": float(
            resolution_1976_result["auc_ci_lower"]
        ),
        "ci_upper": float(
            resolution_1976_result["auc_ci_upper"]
        ),
        "p_value": np.nan,
        "interpretation": (
            "Strong performance remains at fixed resolution"
        ),
    },
    {
        "evidence_domain": "Patient-level information",
        "analysis": "Between-patient score component",
        "sample": "100 patients / 200 eyes",
        "metric": "AUC",
        "estimate": between_patient_component_auc,
        "ci_lower": between_patient_auc_ci["lower"],
        "ci_upper": between_patient_auc_ci["upper"],
        "p_value": np.nan,
        "interpretation": (
            "A large fraction of predictability is patient-level"
        ),
    },
    {
        "evidence_domain": "Eye locality",
        "analysis": "Overall own-eye minus fellow-eye",
        "sample": "100 patients / 200 target eyes",
        "metric": "AUC difference",
        "estimate": observed_auc_difference,
        "ci_lower": overall_bootstrap["difference_lower"],
        "ci_upper": overall_bootstrap["difference_upper"],
        "p_value": np.nan,
        "interpretation": (
            "No significant overall own-eye advantage"
        ),
    },
    {
        "evidence_domain": "Eye locality",
        "analysis": (
            "Symmetric paired-difference ordinal probe"
        ),
        "sample": (
            f"{number_of_validation_pairs} unequal-grade patients"
        ),
        "metric": "Pairwise accuracy",
        "estimate": validation_pair_accuracy,
        "ci_lower": exact_accuracy_ci_lower,
        "ci_upper": exact_accuracy_ci_upper,
        "p_value": float(exact_binomial_result.pvalue),
        "interpretation": (
            "Significant eye-local severity-direction information"
        ),
    },
    {
        "evidence_domain": "Eye locality",
        "analysis": (
            "Symmetric paired-difference ordinal probe"
        ),
        "sample": (
            f"{number_of_validation_pairs} unequal-grade patients"
        ),
        "metric": "Directional AUC",
        "estimate": validation_pair_auc,
        "ci_lower": np.nan,
        "ci_upper": np.nan,
        "p_value": np.nan,
        "interpretation": (
            "Eye-local severity direction is rank-informative"
        ),
    },
]

final_evidence_table = pd.DataFrame(evidence_rows)

display(final_evidence_table)


# ------------------------------------------------------------
# 4. Decision gates
# ------------------------------------------------------------

diagnostic_predictability_gate = bool(
    validation_auc >= 0.80
    and validation_auc_ci["lower"] > 0.50
)

coarse_acquisition_control_gate = bool(
    float(nicheng_result["auc"]) >= 0.80
    and float(resolution_1736_result["auc"]) >= 0.80
    and float(resolution_1976_result["auc"]) >= 0.80
)

eye_locality_gate = bool(
    validation_pair_accuracy > 0.50
    and exact_accuracy_ci_lower > 0.50
    and exact_binomial_result.pvalue < 0.05
)

external_replication_gate = False

if (
    diagnostic_predictability_gate
    and coarse_acquisition_control_gate
    and eye_locality_gate
):
    baseline_decision = (
        "PASS_BASELINE_ADVANCE_TO_LOCALITY_KILL_TEST"
    )
else:
    baseline_decision = (
        "DO_NOT_ADVANCE_REVISE_BASELINE"
    )


decision_summary = {
    "decision": baseline_decision,
    "diagnostic_predictability_gate": (
        diagnostic_predictability_gate
    ),
    "coarse_acquisition_control_gate": (
        coarse_acquisition_control_gate
    ),
    "eye_locality_gate": eye_locality_gate,
    "external_replication_gate": (
        external_replication_gate
    ),
    "primary_locality_result": {
        "correct_pairs": int(correct_pairs),
        "total_pairs": int(
            number_of_validation_pairs
        ),
        "accuracy": float(
            validation_pair_accuracy
        ),
        "exact_two_sided_95ci": [
            exact_accuracy_ci_lower,
            exact_accuracy_ci_upper,
        ],
        "balanced_accuracy": float(
            validation_pair_balanced_accuracy
        ),
        "directional_auc": float(
            validation_pair_auc
        ),
        "exact_one_sided_binomial_p": float(
            exact_binomial_result.pvalue
        ),
    },
    "bounded_conclusion": (
        "Within DeepDRiD, a frozen ImageNet-pretrained "
        "ResNet-50 representation contains strong patient-level "
        "DR information and statistically detectable eye-local "
        "severity-direction information."
    ),
    "claims_not_yet_supported": [
        "Universal cross-dataset generalisation",
        "A cross-modal diagnostic observability law",
        "Complete removal of acquisition confounding",
        "Clinical deployment",
        "Superiority over biomedical foundation models",
    ],
    "next_gate": (
        "Locality kill test followed by independent replication"
    ),
}


# ------------------------------------------------------------
# 5. Environment and artifact manifest
# ------------------------------------------------------------

def sha256_file(file_path):
    file_path = Path(file_path)

    digest = hashlib.sha256()

    with file_path.open("rb") as file:
        while True:
            block = file.read(1024 * 1024)

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


key_artifact_paths = [
    EMBEDDING_PATH,
    EMBEDDING_INDEX_PATH,
    EYE_EMBEDDING_PATH,
    EYE_INDEX_PATH,
    MODEL_PATH,
    SUMMARY_PATH,
    STRATIFIED_METRICS_PATH,
    DECOMPOSITION_SUMMARY_PATH,
    LOCALITY_SUMMARY_PATH,
    PAIRED_MODEL_PATH,
    PAIRED_SUMMARY_PATH,
]

artifact_manifest_rows = []

for artifact_path in key_artifact_paths:
    artifact_path = Path(artifact_path)

    assert artifact_path.is_file(), artifact_path

    artifact_manifest_rows.append(
        {
            "filename": artifact_path.name,
            "path": str(artifact_path),
            "size_bytes": artifact_path.stat().st_size,
            "sha256": sha256_file(artifact_path),
        }
    )

artifact_manifest = pd.DataFrame(
    artifact_manifest_rows
)

environment_summary = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "pytorch": torch.__version__,
    "torchvision": torchvision.__version__,
    "device": str(device),
    "gpu": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    ),
    "random_seed": RANDOM_SEED,
    "encoder": "ResNet-50 / ImageNet-1K V2",
    "encoder_trainable_parameters": 0,
}


# ------------------------------------------------------------
# 6. Save final records
# ------------------------------------------------------------

FINAL_EVIDENCE_PATH = (
    OUTPUT_ROOT
    / "Frozen_Representation_Baseline_v0.1_Evidence_Table.csv"
)

FINAL_DECISION_PATH = (
    OUTPUT_ROOT
    / "Frozen_Representation_Baseline_v0.1_Decision.json"
)

ENVIRONMENT_PATH = (
    OUTPUT_ROOT
    / "Frozen_Representation_Baseline_v0.1_Environment.json"
)

ARTIFACT_MANIFEST_PATH = (
    OUTPUT_ROOT
    / "Frozen_Representation_Baseline_v0.1_Artifact_Manifest.csv"
)

FINAL_REPORT_PATH = (
    OUTPUT_ROOT
    / "Frozen_Representation_Baseline_v0.1_Results_and_Decision.md"
)

final_evidence_table.to_csv(
    FINAL_EVIDENCE_PATH,
    index=False,
)

artifact_manifest.to_csv(
    ARTIFACT_MANIFEST_PATH,
    index=False,
)

with open(
    FINAL_DECISION_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        decision_summary,
        file,
        indent=2,
    )

with open(
    ENVIRONMENT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        environment_summary,
        file,
        indent=2,
    )


report_text = f"""# Retinal DR Frozen Representation Baseline v0.1

**Decision:** `{baseline_decision}`

## Primary findings

- Official validation eye-level AUC: {validation_auc:.4f}
- Between-patient component AUC: {between_patient_component_auc:.4f}
- Main-source Nicheng AUC: {float(nicheng_result["auc"]):.4f}
- Fixed-resolution 1736x1824 AUC: {float(resolution_1736_result["auc"]):.4f}
- Fixed-resolution 1976x1984 AUC: {float(resolution_1976_result["auc"]):.4f}

## Primary eye-locality result

A symmetric paired-difference model was trained on the difference between
left-eye and right-eye frozen embeddings. Model selection used training
patients only.

- Validation unequal-grade patients: {number_of_validation_pairs}
- Correct severity directions: {correct_pairs}/{number_of_validation_pairs}
- Pairwise accuracy: {validation_pair_accuracy:.4f}
- Exact two-sided 95% CI: [{exact_accuracy_ci_lower:.4f}, {exact_accuracy_ci_upper:.4f}]
- Balanced accuracy: {validation_pair_balanced_accuracy:.4f}
- Directional AUC: {validation_pair_auc:.4f}
- Exact one-sided binomial p-value: {exact_binomial_result.pvalue:.6f}

## Interpretation

Within DeepDRiD, the frozen ImageNet-pretrained ResNet-50 representation
contains strong patient-level diabetic-retinopathy information. After
removing patient-shared information through a paired left-minus-right
representation, statistically detectable eye-local severity-direction
information remains.

## Boundaries

This experiment does not yet establish universal cross-dataset
generalisability, a cross-modal law, complete removal of acquisition
confounding, clinical utility, or superiority over biomedical foundation
models.

## Next decision gate

Proceed to a retinal locality kill test, followed by independent dataset
replication before any broad claim.
"""

with open(
    FINAL_REPORT_PATH,
    "w",
    encoding="utf-8",
) as file:
    file.write(report_text)


# ------------------------------------------------------------
# 7. Final output
# ------------------------------------------------------------

print(
    "================ FINAL BASELINE DECISION "
    "================"
)

print("Decision:", baseline_decision)

print(
    "\nDiagnostic predictability gate:",
    diagnostic_predictability_gate,
)

print(
    "Coarse acquisition-control gate:",
    coarse_acquisition_control_gate,
)

print(
    "Eye-locality gate:",
    eye_locality_gate,
)

print(
    "External replication gate:",
    external_replication_gate,
)

print(
    "\nPrimary locality result:"
)

print(
    f"  {correct_pairs}/{number_of_validation_pairs} correct"
)

print(
    "  Accuracy: "
    f"{validation_pair_accuracy:.4f}"
)

print(
    "  Exact two-sided 95% CI: "
    f"[{exact_accuracy_ci_lower:.4f}, "
    f"{exact_accuracy_ci_upper:.4f}]"
)

print(
    "  Exact one-sided p: "
    f"{exact_binomial_result.pvalue:.6f}"
)

print("\nSaved:")
print(FINAL_EVIDENCE_PATH)
print(FINAL_DECISION_PATH)
print(ENVIRONMENT_PATH)
print(ARTIFACT_MANIFEST_PATH)
print(FINAL_REPORT_PATH)

print(
    "\nFrozen representation baseline v0.1 finalized."
)

,evidence_domain,analysis,sample,metric,estimate,ci_lower,ci_upper,p_value,interpretation
0,Diagnostic predictability,Official held-out eye-level linear probe,100 patients / 200 eyes,AUC,0.939192,0.896764,0.973758,NaN,Strong held-out diagnostic predictability
1,Acquisition control,Nicheng source only,92 patients / 184 eyes,AUC,0.934601,0.884967,0.973570,NaN,Strong performance remains within the main source
2,Acquisition control,Fixed resolution 1736x1824,73 patients / 142 eyes,AUC,0.920656,0.850804,0.974640,NaN,Strong performance remains at fixed resolution
3,Acquisition control,Fixed resolution 1976x1984,27 patients / 54 eyes,AUC,0.920635,0.832640,0.980288,NaN,Strong performance remains at fixed resolution
4,Patient-level information,Between-patient score component,100 patients / 200 eyes,AUC,0.945657,0.904131,0.976542,NaN,A large fraction of predictability is patient-...
5,Eye locality,Overall own-eye minus fellow-eye,100 patients / 200 target eyes,AUC difference,0.008182,-0.026970,0.039890,NaN,No significant overall own-eye advantage
6,Eye locality,Symmetric paired-difference ordinal probe,45 unequal-grade patients,Pairwise accuracy,0.688889,0.533509,0.818341,0.008047,Significant eye-local severity-direction infor...
7,Eye locality,Symmetric paired-difference ordinal probe,45 unequal-grade patients,Directional AUC,0.753036,NaN,NaN,NaN,Eye-local severity direction is rank-informative


================ FINAL BASELINE DECISION ================
Decision: PASS_BASELINE_ADVANCE_TO_LOCALITY_KILL_TEST

Diagnostic predictability gate: True
Coarse acquisition-control gate: True
Eye-locality gate: True
External replication gate: False

Primary locality result:
  31/45 correct
  Accuracy: 0.6889
  Exact two-sided 95% CI: [0.5335, 0.8183]
  Exact one-sided p: 0.008047

Saved:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Frozen_Representation_Baseline_v0.1/Frozen_Representation_Baseline_v0.1_Evidence_Table.csv
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Frozen_Representation_Baseline_v0.1/Frozen_Representation_Baseline_v0.1_Decision.json
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Frozen_Representation_Baseline_v0.1/Frozen_Representation_Baseline_v0.1_Environment.json
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR